# International geography domain

Refactored WDQS-backed generator for the `geo_international` domain.

This notebook follows the same structure as the finalized movie/cinema domain:

- RU and EN query text;
- clean English constraints;
- gold labels in RU and EN;
- `ask_validator_sparql`;
- `local_validator`;
- `gold_collection_meta`;
- incremental JSONL saving with progress bars.


## Setup and imports


In [37]:
# ============================================================
# International geography domain v2
# ============================================================
# WDQS-backed generation of high-quality RU/EN multi-hop geography QA.
#
# Design goals:
# - RU and EN query text
# - clean English constraints
# - gold labels in RU and EN
# - full/near-full gold from WDQS, with explicit metadata about completeness
# - richer L3-L5 multi-hop / multi-constraint patterns
# - no external API dependency beyond Wikidata/Wikidata Query Service
#
# The generator is registered as both "geo" and "geo_international" for
# backward compatibility with old runners.

import os
import json
import time
import random
import re
from pathlib import Path
from dataclasses import asdict, is_dataclass
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


## Load common helpers


In [38]:
from pathlib import Path as _Path

if "BenchmarkExample" not in globals():
    exec(_Path("common_helpers.py").read_text(encoding="utf-8"), globals())


## Basic helpers


In [39]:
_GI_QID_CACHE: Dict[str, Optional[str]] = {}

def _gi_norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).casefold()

def _gi_escape(s: str) -> str:
    return str(s or "").replace("\\", "\\\\").replace('"', '\\"')

def _gi_qid(label_en: str, label_ru: Optional[str] = None, fallback_qid: Optional[str] = None) -> Optional[str]:
    """
    Resolve entity QID with local in-memory cache.
    English is tried first because this domain is international and constraints are English.
    """
    key = f"{label_en}|{label_ru or ''}|{fallback_qid or ''}"
    if key in _GI_QID_CACHE:
        return _GI_QID_CACHE[key]

    qid = None
    try:
        qid = resolve_qid(label_en, "en")
    except Exception:
        qid = None

    if not qid and label_ru:
        try:
            qid = resolve_qid(label_ru, "ru")
        except Exception:
            qid = None

    qid = qid or fallback_qid
    _GI_QID_CACHE[key] = qid
    return qid

def _gi_uri_to_qid(uri: str) -> Optional[str]:
    try:
        return uri_to_qid(uri)
    except Exception:
        m = re.search(r"Q\d+", str(uri or ""))
        return m.group(0) if m else None

def _gi_rows_from_select(data: dict) -> List[Dict[str, str]]:
    try:
        return rows_from_select(data)
    except Exception:
        rows = []
        for b in data.get("results", {}).get("bindings", []):
            row = {}
            for k, v in b.items():
                row[k] = v.get("value")
            rows.append(row)
        return rows

def _gi_ru_count(n: int, one: str, few: str, many: str) -> str:
    n = abs(int(n))
    if 11 <= (n % 100) <= 14:
        return many
    last = n % 10
    if last == 1:
        return one
    if 2 <= last <= 4:
        return few
    return many

def _gi_example_to_dict(ex):
    if is_dataclass(ex):
        return asdict(ex)
    if hasattr(ex, "__dict__"):
        return dict(ex.__dict__)
    return dict(ex)

def _gi_clean_constraints(c: Dict[str, Any]) -> Dict[str, Any]:
    """
    Keep constraints compact, English and evaluator-friendly.
    """
    banned = {
        "template_id", "template_family", "candidate_labels", "candidate_labels_ru",
        "candidate_labels_en", "seed_qid", "seed_ru", "seed_en",
        "type_qid", "type_ru", "type_en",
    }
    out = {}
    for k, v in dict(c).items():
        if k in banned:
            continue
        if k.endswith("_ru") or k.endswith("_qid") or k.endswith("_qids"):
            continue
        if v is None or v is False or v == "" or v == [] or v == {}:
            continue
        out[k] = v
    return out

def _gi_make_record_key(r: Dict[str, Any]) -> Tuple[str, str]:
    return (
        r.get("query_text_ru", ""),
        json.dumps(r.get("constraints", {}), ensure_ascii=False, sort_keys=True),
    )


## QIDs and domain vocabulary


In [40]:
GEO_TYPES = {
    "mountain": {
        "qid": "Q8502",
        "ru_acc_singular": "гору",
        "ru_acc_few": "горы",
        "ru_acc_many": "гор",
        "en_plural": "mountains",
        "en_singular": "mountain",
        "metric": "elevation",
    },
    "volcano": {
        "qid": "Q8072",
        "ru_acc_singular": "вулкан",
        "ru_acc_few": "вулкана",
        "ru_acc_many": "вулканов",
        "en_plural": "volcanoes",
        "en_singular": "volcano",
        "metric": "elevation",
    },
    "river": {
        "qid": "Q4022",
        "ru_acc_singular": "реку",
        "ru_acc_few": "реки",
        "ru_acc_many": "рек",
        "en_plural": "rivers",
        "en_singular": "river",
        "metric": None,
    },
    "lake": {
        "qid": "Q23397",
        "ru_acc_singular": "озеро",
        "ru_acc_few": "озера",
        "ru_acc_many": "озёр",
        "en_plural": "lakes",
        "en_singular": "lake",
        "metric": "area",
    },
    "waterfall": {
        "qid": "Q34038",
        "ru_acc_singular": "водопад",
        "ru_acc_few": "водопада",
        "ru_acc_many": "водопадов",
        "en_plural": "waterfalls",
        "en_singular": "waterfall",
        "metric": None,
    },
    "desert": {
        "qid": "Q8514",
        "ru_acc_singular": "пустыню",
        "ru_acc_few": "пустыни",
        "ru_acc_many": "пустынь",
        "en_plural": "deserts",
        "en_singular": "desert",
        "metric": None,
    },
    "island": {
        "qid": "Q23442",
        "ru_acc_singular": "остров",
        "ru_acc_few": "острова",
        "ru_acc_many": "островов",
        "en_plural": "islands",
        "en_singular": "island",
        "metric": "area",
    },
    "sea": {
        "qid": "Q165",
        "ru_acc_singular": "море",
        "ru_acc_few": "моря",
        "ru_acc_many": "морей",
        "en_plural": "seas",
        "en_singular": "sea",
        "metric": None,
    },
}

CONTINENTS = [
    {"en": "Africa", "ru": "Африка", "prep_ru": "в Африке", "qid": "Q15"},
    {"en": "Asia", "ru": "Азия", "prep_ru": "в Азии", "qid": "Q48"},
    {"en": "Europe", "ru": "Европа", "prep_ru": "в Европе", "qid": "Q46"},
    {"en": "North America", "ru": "Северная Америка", "prep_ru": "в Северной Америке", "qid": "Q49"},
    {"en": "South America", "ru": "Южная Америка", "prep_ru": "в Южной Америке", "qid": "Q18"},
    {"en": "Oceania", "ru": "Океания", "prep_ru": "в Океании", "qid": "Q538"},
    {"en": "Antarctica", "ru": "Антарктида", "prep_ru": "в Антарктиде", "qid": "Q51"},
]

COUNTRIES = [
    {"en": "Japan", "ru": "Япония", "prep_ru": "в Японии", "qid": "Q17"},
    {"en": "Italy", "ru": "Италия", "prep_ru": "в Италии", "qid": "Q38"},
    {"en": "Indonesia", "ru": "Индонезия", "prep_ru": "в Индонезии", "qid": "Q252"},
    {"en": "Nepal", "ru": "Непал", "prep_ru": "в Непале", "qid": "Q837"},
    {"en": "Switzerland", "ru": "Швейцария", "prep_ru": "в Швейцарии", "qid": "Q39"},
    {"en": "Canada", "ru": "Канада", "prep_ru": "в Канаде", "qid": "Q16"},
    {"en": "United States", "ru": "США", "prep_ru": "в США", "qid": "Q30"},
    {"en": "Australia", "ru": "Австралия", "prep_ru": "в Австралии", "qid": "Q408"},
    {"en": "Peru", "ru": "Перу", "prep_ru": "в Перу", "qid": "Q419"},
    {"en": "Chile", "ru": "Чили", "prep_ru": "в Чили", "qid": "Q298"},
    {"en": "China", "ru": "Китай", "prep_ru": "в Китае", "qid": "Q148"},
    {"en": "Russia", "ru": "Россия", "prep_ru": "в России", "qid": "Q159"},
    {"en": "Brazil", "ru": "Бразилия", "prep_ru": "в Бразилии", "qid": "Q155"},
    {"en": "Argentina", "ru": "Аргентина", "prep_ru": "в Аргентине", "qid": "Q414"},
    {"en": "Iceland", "ru": "Исландия", "prep_ru": "в Исландии", "qid": "Q189"},
    {"en": "Norway", "ru": "Норвегия", "prep_ru": "в Норвегии", "qid": "Q20"},
    {"en": "India", "ru": "Индия", "prep_ru": "в Индии", "qid": "Q668"},
    {"en": "Mexico", "ru": "Мексика", "prep_ru": "в Мексике", "qid": "Q96"},
    {"en": "New Zealand", "ru": "Новая Зеландия", "prep_ru": "в Новой Зеландии", "qid": "Q664"},
    {"en": "South Africa", "ru": "ЮАР", "prep_ru": "в ЮАР", "qid": "Q258"},
]

# Seeds used for hidden bridge patterns.
GEO_SEEDS = [
    {"kind": "mountain", "en": "Mount Everest", "ru": "Эверест", "qid": "Q513"},
    {"kind": "mountain", "en": "Mont Blanc", "ru": "Монблан", "qid": "Q583"},
    {"kind": "mountain", "en": "Matterhorn", "ru": "Маттерхорн", "qid": "Q2247"},
    {"kind": "volcano", "en": "Mount Fuji", "ru": "Фудзи", "qid": "Q39231"},
    {"kind": "volcano", "en": "Mount Etna", "ru": "Этна", "qid": "Q16976"},
    {"kind": "river", "en": "Nile", "ru": "Нил", "qid": "Q3392"},
    {"kind": "river", "en": "Amazon River", "ru": "Амазонка", "qid": "Q3783"},
    {"kind": "river", "en": "Danube", "ru": "Дунай", "qid": "Q1653"},
    {"kind": "lake", "en": "Lake Victoria", "ru": "Виктория", "qid": "Q5505"},
    {"kind": "lake", "en": "Lake Baikal", "ru": "Байкал", "qid": "Q5513"},
    {"kind": "desert", "en": "Sahara", "ru": "Сахара", "qid": "Q6583"},
    {"kind": "desert", "en": "Gobi Desert", "ru": "Гоби", "qid": "Q42070"},
    {"kind": "island", "en": "Greenland", "ru": "Гренландия", "qid": "Q223"},
    {"kind": "island", "en": "Sicily", "ru": "Сицилия", "qid": "Q1460"},
    {"kind": "sea", "en": "Mediterranean Sea", "ru": "Средиземное море", "qid": "Q4918"},
    {"kind": "sea", "en": "Caribbean Sea", "ru": "Карибское море", "qid": "Q1247"},
    {"kind": "waterfall", "en": "Victoria Falls", "ru": "Виктория", "qid": "Q43278"},
    {"kind": "waterfall", "en": "Niagara Falls", "ru": "Ниагарский водопад", "qid": "Q1287"},
]

def _gi_seed_qid(seed: Dict[str, Any]) -> Optional[str]:
    return seed.get("qid") or _gi_qid(seed["en"], seed.get("ru"))

def _gi_kind_phrase_ru(kind: str, n: int) -> str:
    cfg = GEO_TYPES[kind]
    return _gi_ru_count(n, cfg["ru_acc_singular"], cfg["ru_acc_few"], cfg["ru_acc_many"])

def _gi_kind_phrase_en(kind: str, n: int) -> str:
    return GEO_TYPES[kind]["en_singular"] if int(n) == 1 else GEO_TYPES[kind]["en_plural"]


## SPARQL execution and record finalization


In [41]:
GEO_WDQS_LIMIT_DEFAULT = 701
GEO_MIN_GOLD_BY_LEVEL = {"L1": 3, "L2": 3, "L3": 3, "L4": 3, "L5": 3}
GEO_MAX_GOLD_BY_LEVEL = {"L1": 120, "L2": 90, "L3": 50, "L4": 45, "L5": 35}

def _gi_build_select_query(
    where_lines: List[str],
    *,
    answer_var: str = "item",
    limit: int = GEO_WDQS_LIMIT_DEFAULT,
    extra_select_vars: Optional[List[str]] = None,
) -> str:
    extra_select = " ".join(extra_select_vars or [])
    where = "\n      ".join(where_lines)
    return f"""
    SELECT DISTINCT ?{answer_var} ?{answer_var}LabelEn ?{answer_var}LabelRu {extra_select} WHERE {{
      {where}
      ?{answer_var} rdfs:label ?{answer_var}LabelEn FILTER(LANG(?{answer_var}LabelEn) = "en") .
      OPTIONAL {{ ?{answer_var} rdfs:label ?{answer_var}LabelRu FILTER(LANG(?{answer_var}LabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()

def _gi_build_ask_query(
    where_lines: List[str],
    *,
    answer_var: str = "item",
) -> str:
    ask_lines = []
    for ln in where_lines:
        # Remove label-only / projection-only lines if any.
        if "rdfs:label" in ln:
            continue
        ask_lines.append(ln)
    where = "\n      ".join(ask_lines)
    return f"""
    # WDQS-only validator. All constraints are checked directly in Wikidata.
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{answer_var})
      {where}
    }}
    """.strip()

def _gi_collect_gold(
    *,
    sparql_query: str,
    answer_var: str = "item",
    limit: int = GEO_WDQS_LIMIT_DEFAULT,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    data = wd.sparql_select(sparql_query, use_cache=True)
    rows = _gi_rows_from_select(data)

    seen = set()
    items = []
    dropped_no_qid = 0
    dropped_no_en_label = 0
    ru_label_count = 0
    en_fallback_count = 0

    for row in rows:
        qid = _gi_uri_to_qid(row.get(answer_var))
        if not qid:
            dropped_no_qid += 1
            continue
        if qid in seen:
            continue
        label_en = row.get(f"{answer_var}LabelEn")
        label_ru = row.get(f"{answer_var}LabelRu") or label_en
        if not label_en:
            dropped_no_en_label += 1
            continue
        if row.get(f"{answer_var}LabelRu"):
            ru_label_count += 1
        else:
            en_fallback_count += 1
        seen.add(qid)
        item = {
            "qid": qid,
            "label_en": label_en,
            "label_ru": label_ru,
        }
        # Keep metrics if present.
        for k, v in row.items():
            if k not in {answer_var, f"{answer_var}LabelEn", f"{answer_var}LabelRu"}:
                item[k] = v
        items.append(item)

    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": int(limit),
        "rows_returned_by_wdqs": len(rows),
        "gold_returned_before_limits": len(items),
        "dropped_no_qid_count": dropped_no_qid,
        "dropped_no_en_label_count": dropped_no_en_label,
        "label_sources": {
            "ru_label": ru_label_count,
            "en_fallback_for_ru": en_fallback_count,
        },
        "gold_may_be_incomplete_due_to_wdqs_limit": len(rows) >= int(limit),
    }
    return items, meta

def _gi_quality_filter_gold_items(items: List[Dict[str, Any]], constraints: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Post-filter noisy Wikidata subclass artifacts for user-facing geo tasks.

    WDQS type closure is useful for recall, but for some geo kinds it pulls in
    reefs/banks/rocks as islands and creeks/streams as rivers. For benchmark
    gold we prefer user-facing canonical objects.
    """
    kind = (constraints or {}).get("kind")
    if not kind:
        return items, {"quality_filter_applied": False}

    blacklist_patterns = {
        "island": [
            r"\breef(s)?\b", r"\bbank(s)?\b", r"\brock(s)?\b", r"\bshoal(s)?\b", r"\bsandbank(s)?\b"
        ],
        "river": [
            r"\bcreek\b", r"\bstream\b", r"\bbrook\b", r"\branch\b", r"\bcanal\b", r"\bditch\b"
        ],
        "lake": [
            r"forest park", r"natural reserve", r"nature reserve", r"reservoir region",
            r"provincial natural reserve", r"\bplaya\b", r"\bshuiku\b", r"\breservoir region\b"
        ],
    }

    pats = blacklist_patterns.get(kind, [])
    if not pats:
        return items, {"quality_filter_applied": False}

    kept = []
    dropped = []
    rx = re.compile("|".join(pats), flags=re.IGNORECASE)
    for it in items:
        label_blob = " ".join([str(it.get("label_en") or ""), str(it.get("label_ru") or "")])
        if rx.search(label_blob):
            dropped.append({"qid": it.get("qid"), "label_en": it.get("label_en"), "reason": "label_blacklist"})
        else:
            kept.append(it)

    return kept, {
        "quality_filter_applied": True,
        "quality_filter_kind": kind,
        "quality_filter_label_blacklist": pats,
        "quality_filter_dropped_count": len(dropped),
        "quality_filter_dropped_preview": dropped[:30],
    }



def _gi_has_numeric_or_related_metric(constraints: Dict[str, Any]) -> bool:
    """Return True if a record has a real numeric constraint.

    For L3+ we require at least one answer-side numeric metric or a related-entity
    numeric metric. This prevents weak hard examples such as "waterfalls in one
    of the countries of X" without any further narrowing.
    """
    c = constraints or {}
    metric_keys = {
        "elevation_min_m", "elevation_max_m",
        "area_min_sqkm", "area_max_sqkm",
        "length_min_km", "length_max_km",
        "height_min_m", "height_max_m",
    }
    if any(k in c for k in metric_keys):
        return True
    return any(k.startswith("related_") and (k.endswith("_m") or k.endswith("_km") or k.endswith("_sqkm")) for k in c)

def _gi_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    q_en: str,
    constraints: Dict[str, Any],
    where_lines: List[str],
    requested_count: int,
    answer_var: str = "item",
    extra_select_vars: Optional[List[str]] = None,
    gold_limit: int = 300,
    bridge_meta: Optional[Dict[str, Any]] = None,
    accept_incomplete_for_low_levels: bool = True,
) -> Optional[BenchmarkExample]:
    clean_constraints = _gi_clean_constraints(constraints)

    # Hard levels must be genuinely multi-constraint: bridge + numeric/related metric.
    # L1/L2 remain simple/basic by design.
    if complexity in {"L3", "L4", "L5"} and not _gi_has_numeric_or_related_metric(clean_constraints):
        return None

    sparql = _gi_build_select_query(
        where_lines,
        answer_var=answer_var,
        limit=GEO_WDQS_LIMIT_DEFAULT,
        extra_select_vars=extra_select_vars,
    )
    ask = _gi_build_ask_query(where_lines, answer_var=answer_var)

    try:
        gold_items, meta = _gi_collect_gold(
            sparql_query=sparql,
            answer_var=answer_var,
            limit=GEO_WDQS_LIMIT_DEFAULT,
        )
    except Exception as e:
        return None

    gold_items, quality_meta = _gi_quality_filter_gold_items(gold_items, clean_constraints)
    meta.update(quality_meta)

    n = len(gold_items)
    if n < max(GEO_MIN_GOLD_BY_LEVEL.get(complexity, 3), requested_count):
        return None

    # Reject very broad high-level examples. This keeps L4/L5 selective.
    if n > GEO_MAX_GOLD_BY_LEVEL.get(complexity, 300):
        return None

    # For L3-L5 prefer not to accept results that hit the WDQS limit.
    if complexity in {"L3", "L4", "L5"} and meta.get("gold_may_be_incomplete_due_to_wdqs_limit"):
        return None

    truncated_by_local_limit = n > gold_limit
    final_items = gold_items[:gold_limit]

    meta.update({
        "constraints_are_wdqs_only": True,
        "gold_limit": int(gold_limit),
        "gold_returned": len(final_items),
        "gold_total_before_limit": n,
        "gold_truncated_by_local_limit": truncated_by_local_limit,
        "template_id": template_id,
        "template_family": template_family,
    })
    if bridge_meta:
        meta["bridge_meta"] = bridge_meta

    local_validator = {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": {**clean_constraints, **({"quality_label_blacklist": meta.get("quality_filter_label_blacklist")} if meta.get("quality_filter_applied") else {})},
        "label_matching_used": False,
        "note": "All constraints for this geography task are represented in the WDQS ASK validator; no external local validator is required.",
    }

    return BenchmarkExample(
        id=f"geo_int_{complexity.lower()}_{idx:04d}",
        domain="geo_international",
        complexity=complexity,
        query_text_ru=q_ru,
        query_text_en=q_en,
        constraints=clean_constraints,
        requested_count=int(requested_count),
        gold_answer_qids=[it["qid"] for it in final_items],
        gold_answer_labels_ru=[it["label_ru"] for it in final_items],
        gold_answer_labels_en=[it["label_en"] for it in final_items],
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=(complexity in {"L3", "L4", "L5"}),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=truncated_by_local_limit,
        ask_validator_sparql=ask,
        local_validator=local_validator,
        gold_collection_meta=meta,
    )


## SPARQL WHERE builders


In [42]:
GEO_DIRECT_TYPE_KINDS = {"river", "island", "lake"}

# Unit QIDs used by Wikidata quantity statements.
# We normalize numeric values so constraints like area_min_sqkm really mean km².
WD_UNIT_METRE = "Q11573"
WD_UNIT_KILOMETRE = "Q828224"
WD_UNIT_FOOT = "Q3710"
WD_UNIT_MILE = "Q253276"
WD_UNIT_SQ_KILOMETRE = "Q712226"
WD_UNIT_SQ_METRE = "Q25343"
WD_UNIT_HECTARE = "Q35852"
WD_UNIT_SQ_MILE = "Q232291"


def _gi_type_line(kind: str) -> str:
    # For river/island/lake, direct P31 avoids noisy subclass artifacts such as
    # creeks/streams, reefs/banks/rocks, and parks/reserves/reservoir regions.
    if kind in GEO_DIRECT_TYPE_KINDS:
        return f"?item wdt:P31 wd:{GEO_TYPES[kind]['qid']} ."
    return f"?item wdt:P31/wdt:P279* wd:{GEO_TYPES[kind]['qid']} ."


def _gi_quantity_value_lines(
    subject_var: str,
    prop: str,
    out_var: str,
    prefix: str,
    quantity_kind: str,
) -> List[str]:
    """Read a Wikidata quantity through psv and normalize units.

    Using wdt:P2046 / wdt:P2043 / wdt:P2044 directly can silently mix units.
    These builders normalize:
    - area -> square kilometres
    - length -> kilometres
    - elevation/height -> metres
    """
    stmt = f"{prefix}Statement"
    value = f"{prefix}Value"
    amount = f"{prefix}Amount"
    unit = f"{prefix}Unit"

    lines = [
        f"?{subject_var} p:{prop} ?{stmt} .",
        f"?{stmt} psv:{prop} ?{value} .",
        f"?{value} wikibase:quantityAmount ?{amount} .",
        f"?{value} wikibase:quantityUnit ?{unit} .",
    ]

    if quantity_kind == "area_sqkm":
        expr = (
            f"IF(?{unit} = wd:{WD_UNIT_SQ_KILOMETRE}, xsd:decimal(?{amount}), "
            f"IF(?{unit} = wd:{WD_UNIT_SQ_METRE}, xsd:decimal(?{amount}) / 1000000, "
            f"IF(?{unit} = wd:{WD_UNIT_HECTARE}, xsd:decimal(?{amount}) / 100, "
            f"IF(?{unit} = wd:{WD_UNIT_SQ_MILE}, xsd:decimal(?{amount}) * 2.589988110336, xsd:decimal(?{amount})))))"
        )
    elif quantity_kind == "length_km":
        expr = (
            f"IF(?{unit} = wd:{WD_UNIT_KILOMETRE}, xsd:decimal(?{amount}), "
            f"IF(?{unit} = wd:{WD_UNIT_METRE}, xsd:decimal(?{amount}) / 1000, "
            f"IF(?{unit} = wd:{WD_UNIT_MILE}, xsd:decimal(?{amount}) * 1.609344, "
            f"IF(?{unit} = wd:{WD_UNIT_FOOT}, xsd:decimal(?{amount}) * 0.0003048, xsd:decimal(?{amount})))))"
        )
    elif quantity_kind == "elevation_m":
        expr = (
            f"IF(?{unit} = wd:{WD_UNIT_METRE}, xsd:decimal(?{amount}), "
            f"IF(?{unit} = wd:{WD_UNIT_KILOMETRE}, xsd:decimal(?{amount}) * 1000, "
            f"IF(?{unit} = wd:{WD_UNIT_FOOT}, xsd:decimal(?{amount}) * 0.3048, xsd:decimal(?{amount}))))"
        )
    else:
        expr = f"xsd:decimal(?{amount})"

    lines.append(f"BIND(({expr}) AS ?{out_var}) .")
    return lines


def _gi_metric_value_lines(subject_var: str, metric: str, out_var: str, prefix: str) -> List[str]:
    if metric == "elevation_m":
        return _gi_quantity_value_lines(subject_var, "P2044", out_var, prefix, "elevation_m")
    if metric == "area_sqkm":
        return _gi_quantity_value_lines(subject_var, "P2046", out_var, prefix, "area_sqkm")
    if metric == "length_km":
        return _gi_quantity_value_lines(subject_var, "P2043", out_var, prefix, "length_km")
    raise ValueError(f"Unknown metric: {metric}")


def _gi_has_en_label_line(answer_var: str = "item") -> str:
    return f'?{answer_var} rdfs:label ?{answer_var}LabelEn FILTER(LANG(?{answer_var}LabelEn) = "en") .'


def _gi_country_line(country: Dict[str, str]) -> str:
    return f"?item wdt:P17 wd:{country['qid']} ."


def _gi_continent_line(continent: Dict[str, str]) -> str:
    return f"?item wdt:P30 wd:{continent['qid']} ."


def _gi_country_on_continent_lines(country: Dict[str, str], continent: Dict[str, str]) -> List[str]:
    return [
        f"?item wdt:P17 wd:{country['qid']} .",
        f"wd:{country['qid']} wdt:P30 wd:{continent['qid']} .",
    ]


def _gi_elevation_min_lines(min_m: int) -> List[str]:
    return _gi_metric_value_lines("item", "elevation_m", "elevation_m", "itemElevation") + [
        f"FILTER(?elevation_m >= {int(min_m)}) .",
    ]


def _gi_elevation_max_lines(max_m: int) -> List[str]:
    return _gi_metric_value_lines("item", "elevation_m", "elevation_m", "itemElevation") + [
        f"FILTER(?elevation_m <= {int(max_m)}) .",
    ]


def _gi_elevation_range_lines(min_m: int, max_m: int) -> List[str]:
    return _gi_metric_value_lines("item", "elevation_m", "elevation_m", "itemElevation") + [
        f"FILTER(?elevation_m >= {int(min_m)} && ?elevation_m <= {int(max_m)}) .",
    ]


def _gi_area_min_lines(min_sqkm: int) -> List[str]:
    return _gi_metric_value_lines("item", "area_sqkm", "area_sqkm", "itemArea") + [
        f"FILTER(?area_sqkm >= {int(min_sqkm)}) .",
    ]


def _gi_area_max_lines(max_sqkm: int) -> List[str]:
    return _gi_metric_value_lines("item", "area_sqkm", "area_sqkm", "itemArea") + [
        f"FILTER(?area_sqkm <= {int(max_sqkm)}) .",
    ]


def _gi_area_range_lines(min_sqkm: int, max_sqkm: int) -> List[str]:
    return _gi_metric_value_lines("item", "area_sqkm", "area_sqkm", "itemArea") + [
        f"FILTER(?area_sqkm >= {int(min_sqkm)} && ?area_sqkm <= {int(max_sqkm)}) .",
    ]


def _gi_length_min_lines(min_km: int) -> List[str]:
    return _gi_metric_value_lines("item", "length_km", "length_km", "itemLength") + [
        f"FILTER(?length_km >= {int(min_km)}) .",
    ]


def _gi_length_max_lines(max_km: int) -> List[str]:
    return _gi_metric_value_lines("item", "length_km", "length_km", "itemLength") + [
        f"FILTER(?length_km <= {int(max_km)}) .",
    ]


def _gi_length_range_lines(min_km: int, max_km: int) -> List[str]:
    return _gi_metric_value_lines("item", "length_km", "length_km", "itemLength") + [
        f"FILTER(?length_km >= {int(min_km)} && ?length_km <= {int(max_km)}) .",
    ]


## RU/EN query text builders


In [43]:
def _gi_nlg_head(kind: str, requested_count: int) -> Tuple[str, str]:
    ru_noun = _gi_kind_phrase_ru(kind, requested_count)
    en_noun = _gi_kind_phrase_en(kind, requested_count)
    return f"Назови {requested_count} {ru_noun}", f"Name {requested_count} {en_noun}"

def _gi_join_ru(parts: List[str]) -> str:
    return ", ".join(p for p in parts if p)

def _gi_join_en(parts: List[str]) -> str:
    return ", ".join(p for p in parts if p)

def _gi_finalize_text(kind: str, requested_count: int, ru_parts: List[str], en_parts: List[str]) -> Tuple[str, str]:
    head_ru, head_en = _gi_nlg_head(kind, requested_count)
    return head_ru + ", " + _gi_join_ru(ru_parts) + ".", head_en + " " + _gi_join_en(en_parts) + "."

def _gi_country_phrase_ru(country: Dict[str, str]) -> str:
    return country.get("prep_ru") or f"в стране {country['ru']}"

def _gi_continent_phrase_ru(continent: Dict[str, str]) -> str:
    return continent.get("prep_ru") or f"на континенте {continent['ru']}"

def _gi_metric_phrase_ru(c: Dict[str, Any]) -> List[str]:
    parts = []
    if c.get("elevation_min_m") is not None and c.get("elevation_max_m") is not None:
        parts.append(f"с высотой от {int(c['elevation_min_m'])} до {int(c['elevation_max_m'])} м")
    elif c.get("elevation_min_m") is not None:
        parts.append(f"с высотой не ниже {int(c['elevation_min_m'])} м")
    elif c.get("elevation_max_m") is not None:
        parts.append(f"с высотой не выше {int(c['elevation_max_m'])} м")

    if c.get("area_min_sqkm") is not None and c.get("area_max_sqkm") is not None:
        parts.append(f"с площадью от {int(c['area_min_sqkm'])} до {int(c['area_max_sqkm'])} км²")
    elif c.get("area_min_sqkm") is not None:
        parts.append(f"с площадью не менее {int(c['area_min_sqkm'])} км²")
    elif c.get("area_max_sqkm") is not None:
        parts.append(f"с площадью не более {int(c['area_max_sqkm'])} км²")

    if c.get("length_min_km") is not None and c.get("length_max_km") is not None:
        parts.append(f"с длиной от {int(c['length_min_km'])} до {int(c['length_max_km'])} км")
    elif c.get("length_min_km") is not None:
        parts.append(f"с длиной не менее {int(c['length_min_km'])} км")
    elif c.get("length_max_km") is not None:
        parts.append(f"с длиной не более {int(c['length_max_km'])} км")
    return parts

def _gi_metric_phrase_en(c: Dict[str, Any]) -> List[str]:
    parts = []
    if c.get("elevation_min_m") is not None and c.get("elevation_max_m") is not None:
        parts.append(f"with elevation from {int(c['elevation_min_m'])} to {int(c['elevation_max_m'])} m")
    elif c.get("elevation_min_m") is not None:
        parts.append(f"with elevation at least {int(c['elevation_min_m'])} m")
    elif c.get("elevation_max_m") is not None:
        parts.append(f"with elevation at most {int(c['elevation_max_m'])} m")

    if c.get("area_min_sqkm") is not None and c.get("area_max_sqkm") is not None:
        parts.append(f"with area from {int(c['area_min_sqkm'])} to {int(c['area_max_sqkm'])} sq km")
    elif c.get("area_min_sqkm") is not None:
        parts.append(f"with area at least {int(c['area_min_sqkm'])} sq km")
    elif c.get("area_max_sqkm") is not None:
        parts.append(f"with area at most {int(c['area_max_sqkm'])} sq km")

    if c.get("length_min_km") is not None and c.get("length_max_km") is not None:
        parts.append(f"with length from {int(c['length_min_km'])} to {int(c['length_max_km'])} km")
    elif c.get("length_min_km") is not None:
        parts.append(f"with length at least {int(c['length_min_km'])} km")
    elif c.get("length_max_km") is not None:
        parts.append(f"with length at most {int(c['length_max_km'])} km")
    return parts


## Template builders


In [44]:
def _gi_tpl_country_kind(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None

    # Broad river/island country queries are noisy in Wikidata. Use them only
    # with length/area constraints from L2 upward.
    if complexity == "L1":
        kind = rng.choice(["mountain", "volcano", "lake", "waterfall", "desert"])
    elif complexity == "L2":
        kind = rng.choice(["mountain", "volcano", "lake", "desert", "sea", "river", "island"])
    else:
        kind = rng.choice(["mountain", "volcano", "lake", "desert", "sea", "river", "island"])

    country = rng.choice(COUNTRIES)
    requested = 5 if complexity in {"L1", "L2"} else 4

    where = [_gi_type_line(kind), _gi_country_line(country)]
    constraints = {"kind": kind, "country": country["en"]}

    ru_parts = [f"расположенных {_gi_country_phrase_ru(country)}"]
    en_parts = [f"located in {country['en']}"]
    template_id = "geo_int_country_kind"

    if complexity == "L2" and kind in {"mountain", "volcano"}:
        if rng.random() < 0.45:
            max_m = rng.choice([1000, 1500, 2000, 2500, 3000])
            where += _gi_elevation_max_lines(max_m)
            constraints["elevation_max_m"] = max_m
            template_id += "_elevation_max"
        else:
            min_m = rng.choice([1000, 1500, 2000, 2500, 3000])
            where += _gi_elevation_min_lines(min_m)
            constraints["elevation_min_m"] = min_m
            template_id += "_elevation_min"
    elif complexity == "L2" and kind == "river":
        if rng.random() < 0.40:
            max_len = rng.choice([100, 250, 500, 1000])
            where += _gi_length_max_lines(max_len)
            constraints["length_max_km"] = max_len
            template_id += "_length_max"
        else:
            min_len = rng.choice([50, 100, 250, 500, 1000])
            where += _gi_length_min_lines(min_len)
            constraints["length_min_km"] = min_len
            template_id += "_length_min"
    elif complexity == "L2" and kind in {"lake", "island", "desert", "sea"}:
        if rng.random() < 0.40:
            max_area = rng.choice([50, 100, 500, 1000, 5000])
            where += _gi_area_max_lines(max_area)
            constraints["area_max_sqkm"] = max_area
            template_id += "_area_max"
        else:
            min_area = rng.choice([50, 100, 500, 1000, 5000])
            where += _gi_area_min_lines(min_area)
            constraints["area_min_sqkm"] = min_area
            template_id += "_area_min"

    if complexity == "L2" and not _gi_has_numeric_or_related_metric(constraints):
        return None

    ru_parts += _gi_metric_phrase_ru(constraints)
    en_parts += _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="country_kind",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
    )

def _gi_tpl_continent_kind(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None

    # Avoid noisy broad river/island-by-continent L1 queries. Bring them back
    # only with length/area constraints.
    if complexity == "L1":
        kind = rng.choice(["mountain", "lake", "desert", "sea", "waterfall"])
    elif complexity == "L2":
        kind = rng.choice(["mountain", "volcano", "lake", "desert", "sea", "river", "island"])
    else:
        kind = rng.choice(["mountain", "volcano", "lake", "desert", "sea", "river", "island"])

    continent = rng.choice(CONTINENTS)
    requested = 5 if complexity in {"L1", "L2"} else 4

    where = [_gi_type_line(kind), _gi_continent_line(continent)]
    constraints = {"kind": kind, "continent": continent["en"]}

    ru_parts = [f"расположенных {_gi_continent_phrase_ru(continent)}"]
    en_parts = [f"located in {continent['en']}"]
    template_id = "geo_int_continent_kind"

    if complexity == "L2" and kind == "mountain":
        if rng.random() < 0.40:
            max_m = rng.choice([1000, 2000, 3000, 4000])
            where += _gi_elevation_max_lines(max_m)
            constraints["elevation_max_m"] = max_m
            template_id += "_elevation_max"
        else:
            min_m = rng.choice([2000, 3000, 4000, 5000, 6000])
            where += _gi_elevation_min_lines(min_m)
            constraints["elevation_min_m"] = min_m
            template_id += "_elevation_min"
    elif complexity == "L2" and kind == "river":
        if rng.random() < 0.35:
            max_len = rng.choice([250, 500, 1000, 2000])
            where += _gi_length_max_lines(max_len)
            constraints["length_max_km"] = max_len
            template_id += "_length_max"
        else:
            min_len = rng.choice([250, 500, 1000, 2000])
            where += _gi_length_min_lines(min_len)
            constraints["length_min_km"] = min_len
            template_id += "_length_min"
    elif complexity == "L2" and kind in {"lake", "island", "desert", "sea"}:
        if rng.random() < 0.35:
            max_area = rng.choice([100, 500, 1000, 5000])
            where += _gi_area_max_lines(max_area)
            constraints["area_max_sqkm"] = max_area
            template_id += "_area_max"
        else:
            min_area = rng.choice([500, 1000, 5000, 10000])
            where += _gi_area_min_lines(min_area)
            constraints["area_min_sqkm"] = min_area
            template_id += "_area_min"

    if complexity == "L2" and not _gi_has_numeric_or_related_metric(constraints):
        return None

    ru_parts += _gi_metric_phrase_ru(constraints)
    en_parts += _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id=template_id,
        template_family="continent_kind",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
    )

def _gi_tpl_mountain_range_elevation(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None

    seeds = [s for s in GEO_SEEDS if s["kind"] == "mountain"]
    seed = rng.choice(seeds)
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None

    requested = 3
    if complexity == "L3":
        # Direct range by seed bridge: still multihop, but not too tight.
        min_m = rng.choice([4000, 5000, 6000, 7000])
    elif complexity == "L4":
        min_m = rng.choice([5000, 6000, 7000, 8000])
    else:
        min_m = rng.choice([6000, 7000, 8000])

    where = [
        _gi_type_line("mountain"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P4552 ?bridgeRange .",
        "?item wdt:P4552 ?bridgeRange .",
        "FILTER(?item != ?seed) .",
    ] + _gi_elevation_min_lines(min_m)

    constraints = {
        "kind": "mountain",
        "mountain_range_from_object": seed["en"],
        "elevation_min_m": min_m,
    }

    ru_parts = [
        f"из того же горного хребта, что и «{seed['ru']}»",
        f"с высотой не ниже {min_m} м",
    ]
    en_parts = [
        f"from the same mountain range as \"{seed['en']}\"",
        f"with elevation at least {min_m} m",
    ]
    q_ru, q_en = _gi_finalize_text("mountain", requested, ru_parts, en_parts)

    bridge_meta = {
        "bridge": "mountain_range",
        "constraint_key": "mountain_range_from_object",
        "wikidata_property": "P4552",
        "seed_label_en": seed["en"],
        "seed_label_ru": seed["ru"],
        "semantics": "any_shared_value",
        "intermediate_value_hidden_in_query": True,
    }

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_mountain_same_range_elevation",
        template_family="mountains",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m"],
        bridge_meta=bridge_meta,
    )

def _gi_tpl_same_country_metric(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None

    seed = rng.choice([s for s in GEO_SEEDS if s["kind"] in {"mountain", "volcano", "lake", "island", "waterfall"}])
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None

    # Answer kind can differ from seed kind. This creates more interesting multi-hop.
    if seed["kind"] in {"mountain", "volcano"}:
        kind = rng.choice(["mountain", "volcano", "lake", "waterfall"])
    elif seed["kind"] in {"lake", "waterfall"}:
        kind = rng.choice(["lake", "river", "waterfall", "mountain"])
    elif seed["kind"] == "island":
        kind = rng.choice(["island", "mountain", "lake"])
    else:
        kind = rng.choice(["mountain", "lake", "river"])

    requested = 3

    where = [
        _gi_type_line(kind),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P17 ?bridgeCountry .",
        "?item wdt:P17 ?bridgeCountry .",
        "FILTER(?item != ?seed) .",
    ]

    constraints = {
        "kind": kind,
        "country_from_object": seed["en"],
    }

    # Add reliable numeric constraints for L4/L5 or sometimes L3.
    if kind in {"mountain", "volcano"}:
        if complexity == "L5":
            lo = rng.choice([1000, 1500, 2000, 2500, 3000])
            hi = lo + rng.choice([1000, 1500, 2000, 2500])
            where += _gi_elevation_range_lines(lo, hi)
            constraints["elevation_min_m"] = lo
            constraints["elevation_max_m"] = hi
        else:
            min_m = rng.choice([1000, 1500, 2000, 2500, 3000])
            where += _gi_elevation_min_lines(min_m)
            constraints["elevation_min_m"] = min_m
    elif kind == "river" and complexity in {"L3", "L4", "L5"}:
        min_len = rng.choice([100, 250, 500, 1000])
        where += _gi_length_min_lines(min_len)
        constraints["length_min_km"] = min_len
    elif kind in {"lake", "island"} and complexity in {"L3", "L4", "L5"}:
        min_area = rng.choice([100, 500, 1000, 5000])
        where += _gi_area_min_lines(min_area)
        constraints["area_min_sqkm"] = min_area

    ru_parts = [f"расположенных в одной из стран, где находится «{seed['ru']}»"]
    en_parts = [f"located in one of the countries of \"{seed['en']}\""]
    if complexity == "L2" and not _gi_has_numeric_or_related_metric(constraints):
        return None

    ru_parts += _gi_metric_phrase_ru(constraints)
    en_parts += _gi_metric_phrase_en(constraints)

    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)

    bridge_meta = {
        "bridge": "country",
        "constraint_key": "country_from_object",
        "wikidata_property": "P17",
        "seed_label_en": seed["en"],
        "seed_label_ru": seed["ru"],
        "semantics": "any_shared_value",
        "intermediate_value_hidden_in_query": True,
    }

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_same_country_metric",
        template_family="same_country",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
        bridge_meta=bridge_meta,
    )

def _gi_tpl_river_same_mouth(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None

    seed = rng.choice([s for s in GEO_SEEDS if s["kind"] == "river"])
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None

    requested = 3
    where = [
        _gi_type_line("river"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P403 ?bridgeMouth .",
        "?item wdt:P403 ?bridgeMouth .",
        "FILTER(?item != ?seed) .",
    ]

    constraints = {
        "kind": "river",
        "mouth_body_from_object": seed["en"],
    }

    # L5 adds continent through the answer's own P30 if available.
    ru_parts = [f"впадающих в тот же водоём, что и «{seed['ru']}»"]
    en_parts = [f"flowing into the same body of water as \"{seed['en']}\""]

    if complexity in {"L4", "L5"} and rng.random() < (0.35 if complexity == "L4" else 0.65):
        min_len = rng.choice([100, 250, 500, 1000])
        where += _gi_length_min_lines(min_len)
        constraints["length_min_km"] = min_len
        ru_parts += _gi_metric_phrase_ru(constraints)
        en_parts += _gi_metric_phrase_en(constraints)

    if complexity == "L5" and rng.random() < 0.65:
        cont = rng.choice(CONTINENTS)
        where.append(_gi_continent_line(cont))
        constraints["continent"] = cont["en"]
        ru_parts.append(f"расположенных {_gi_continent_phrase_ru(cont)}")
        en_parts.append(f"located in {cont['en']}")

    q_ru, q_en = _gi_finalize_text("river", requested, ru_parts, en_parts)

    bridge_meta = {
        "bridge": "mouth_body",
        "constraint_key": "mouth_body_from_object",
        "wikidata_property": "P403",
        "seed_label_en": seed["en"],
        "seed_label_ru": seed["ru"],
        "semantics": "any_shared_value",
        "intermediate_value_hidden_in_query": True,
    }

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_river_same_mouth",
        template_family="rivers",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?length_km"],
        bridge_meta=bridge_meta,
    )

def _gi_tpl_same_continent_as_seed(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None

    seed = rng.choice([s for s in GEO_SEEDS if s["kind"] in {"desert", "sea", "lake", "island", "mountain"}])
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None

    kind = rng.choice(["desert", "lake", "island", "mountain", "sea", "river"])
    requested = 3 if complexity in {"L4", "L5"} else 4

    where = [
        _gi_type_line(kind),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P30 ?bridgeContinent .",
        "?item wdt:P30 ?bridgeContinent .",
        "FILTER(?item != ?seed) .",
    ]

    constraints = {
        "kind": kind,
        "continent_from_object": seed["en"],
    }

    if kind == "mountain" and complexity in {"L3", "L4", "L5"}:
        min_m = rng.choice([2000, 3000, 4000, 5000])
        where += _gi_elevation_min_lines(min_m)
        constraints["elevation_min_m"] = min_m
    elif kind in {"lake", "island"} and complexity in {"L3", "L4", "L5"}:
        min_area = rng.choice([500, 1000, 5000, 10000])
        where += _gi_area_min_lines(min_area)
        constraints["area_min_sqkm"] = min_area
    elif kind == "river" and complexity in {"L3", "L4", "L5"}:
        min_len = rng.choice([250, 500, 1000, 2000])
        where += _gi_length_min_lines(min_len)
        constraints["length_min_km"] = min_len

    ru_parts = [f"находящихся на том же континенте, что и «{seed['ru']}»"]
    en_parts = [f"located on the same continent as \"{seed['en']}\""]
    if complexity == "L2" and not _gi_has_numeric_or_related_metric(constraints):
        return None

    ru_parts += _gi_metric_phrase_ru(constraints)
    en_parts += _gi_metric_phrase_en(constraints)

    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)

    bridge_meta = {
        "bridge": "continent",
        "constraint_key": "continent_from_object",
        "wikidata_property": "P30",
        "seed_label_en": seed["en"],
        "seed_label_ru": seed["ru"],
        "semantics": "any_shared_value",
        "intermediate_value_hidden_in_query": True,
    }

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_same_continent_as_seed",
        template_family="same_continent",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
        bridge_meta=bridge_meta,
    )

def _gi_tpl_country_continent_metric(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None

    # Bridge through country -> continent is a clean 2-hop Wikidata pattern.
    country = rng.choice(COUNTRIES)
    kind = rng.choice(["mountain", "volcano", "lake", "island", "river", "waterfall"])
    requested = 3 if complexity in {"L4", "L5"} else 4

    where = [
        _gi_type_line(kind),
        f"?item wdt:P17 wd:{country['qid']} .",
        f"wd:{country['qid']} wdt:P30 ?bridgeContinent .",
        "?item wdt:P30 ?bridgeContinent .",
    ]

    constraints = {
        "kind": kind,
        "country": country["en"],
        "continent_via_country": True,
    }

    if kind in {"mountain", "volcano"}:
        min_m = rng.choice([1000, 1500, 2000, 2500, 3000])
        where += _gi_elevation_min_lines(min_m)
        constraints["elevation_min_m"] = min_m
    elif kind == "river" and complexity in {"L3", "L4", "L5"}:
        min_len = rng.choice([100, 250, 500, 1000])
        where += _gi_length_min_lines(min_len)
        constraints["length_min_km"] = min_len
    elif kind in {"lake", "island"} and complexity in {"L3", "L4", "L5"}:
        min_area = rng.choice([100, 500, 1000, 5000])
        where += _gi_area_min_lines(min_area)
        constraints["area_min_sqkm"] = min_area

    ru_parts = [
        f"расположенных {_gi_country_phrase_ru(country)}",
        f"и на том же континенте, к которому относится эта страна",
    ]
    en_parts = [
        f"located in {country['en']}",
        f"and on the continent of that country",
    ]
    if complexity == "L2" and not _gi_has_numeric_or_related_metric(constraints):
        return None

    ru_parts += _gi_metric_phrase_ru(constraints)
    en_parts += _gi_metric_phrase_en(constraints)

    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)

    bridge_meta = {
        "bridge": "country_to_continent",
        "constraint_key": "continent_via_country",
        "wikidata_property_chain": ["P17", "P30"],
        "country_label_en": country["en"],
        "country_label_ru": country["ru"],
        "semantics": "country_continent_consistency",
    }

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_country_continent_metric",
        template_family="country_continent",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
        bridge_meta=bridge_meta,
    )


## Template registry and generator


## Hard L3-L5 diverse template patch

Extra multi-hop / multi-constraint patterns for the international geography domain.
This patch intentionally increases L3-L5 diversity and keeps broad noisy L1-L2 patterns out of hard levels.


In [45]:

# ============================================================
# Hard L3-L5 diverse template patch
# ============================================================
# This cell adds more multi-hop and multi-constraint templates and replaces
# the registry with a weighted hard-level mix.

GEO_HARD_KIND_TARGETS = {
    "L3": {"mountain": 0.20, "river": 0.20, "lake": 0.16, "island": 0.16, "volcano": 0.10, "desert": 0.08, "sea": 0.05, "waterfall": 0.05},
    "L4": {"mountain": 0.20, "river": 0.22, "lake": 0.15, "island": 0.16, "volcano": 0.10, "desert": 0.07, "sea": 0.05, "waterfall": 0.05},
    "L5": {"mountain": 0.22, "river": 0.23, "lake": 0.14, "island": 0.15, "volcano": 0.11, "desert": 0.07, "sea": 0.04, "waterfall": 0.04},
}


def _gi_pick_weighted(rng: random.Random, items: List[Any], weights: List[float]) -> Any:
    return rng.choices(items, weights=weights, k=1)[0]


def _gi_seed(kind: str, rng: random.Random) -> Optional[Dict[str, Any]]:
    pool = [s for s in GEO_SEEDS if s.get("kind") == kind]
    return rng.choice(pool) if pool else None


def _gi_bridge_meta(
    *,
    bridge: str,
    constraint_key: str,
    property_id: Optional[str] = None,
    property_chain: Optional[List[str]] = None,
    seed: Optional[Dict[str, Any]] = None,
    semantics: str = "any_shared_value",
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    out = {
        "bridge": bridge,
        "constraint_key": constraint_key,
        "semantics": semantics,
        "intermediate_value_hidden_in_query": True,
    }
    if property_id:
        out["wikidata_property"] = property_id
    if property_chain:
        out["wikidata_property_chain"] = property_chain
    if seed:
        out.update({
            "seed_label_en": seed.get("en"),
            "seed_label_ru": seed.get("ru"),
            "seed_kind": seed.get("kind"),
        })
    if extra:
        out.update(extra)
    return out


def _gi_add_metric_for_kind(
    *,
    kind: str,
    complexity: str,
    rng: random.Random,
    where: List[str],
    constraints: Dict[str, Any],
    force: bool = True,
    prefer_range: bool = False,
) -> None:
    """Add a reliable numeric geo constraint for the selected kind when possible."""
    if not force and rng.random() < 0.35:
        return

    if kind in {"mountain", "volcano"}:
        if prefer_range or complexity == "L5":
            lo = rng.choice([1000, 1500, 2000, 2500, 3000, 4000])
            hi = lo + rng.choice([1000, 1500, 2000, 2500, 3000])
            where += _gi_elevation_range_lines(lo, hi)
            constraints["elevation_min_m"] = lo
            constraints["elevation_max_m"] = hi
        else:
            lo = rng.choice([1000, 1500, 2000, 2500, 3000, 4000, 5000])
            where += _gi_elevation_min_lines(lo)
            constraints["elevation_min_m"] = lo
        return

    if kind == "river":
        if prefer_range or complexity == "L5":
            lo = rng.choice([100, 250, 500, 1000])
            hi = lo + rng.choice([500, 1000, 1500, 2500])
            where += _gi_length_range_lines(lo, hi)
            constraints["length_min_km"] = lo
            constraints["length_max_km"] = hi
        else:
            lo = rng.choice([100, 250, 500, 1000, 2000])
            where += _gi_length_min_lines(lo)
            constraints["length_min_km"] = lo
        return

    if kind in {"lake", "island", "desert", "sea"}:
        if prefer_range or complexity == "L5":
            lo = rng.choice([100, 500, 1000, 5000, 10000])
            hi = lo * rng.choice([2, 3, 5, 10])
            where += _gi_area_range_lines(lo, hi)
            constraints["area_min_sqkm"] = lo
            constraints["area_max_sqkm"] = hi
        else:
            lo = rng.choice([100, 500, 1000, 5000, 10000])
            where += _gi_area_min_lines(lo)
            constraints["area_min_sqkm"] = lo
        return


# -------------------------
# Additional templates
# -------------------------

def _gi_tpl_island_same_water_body_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    seed = _gi_seed("island", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 4 if complexity == "L3" else 3
    where = [
        _gi_type_line("island"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P206 ?bridgeWaterBody .",
        "?item wdt:P206 ?bridgeWaterBody .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "island", "water_body_from_object": seed["en"]}
    _gi_add_metric_for_kind(kind="island", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    ru_parts = [f"расположенных в том же водоёме или у того же водоёма, что и «{seed['ru']}»"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"located in or next to the same body of water as \"{seed['en']}\""] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("island", requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_island_same_water_body_area",
        template_family="island_water_body", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(bridge="water_body", constraint_key="water_body_from_object", property_id="P206", seed=seed),
    )


def _gi_tpl_island_same_part_of_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    seed = _gi_seed("island", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 3
    where = [
        _gi_type_line("island"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P361 ?bridgeWhole .",
        "?item wdt:P361 ?bridgeWhole .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "island", "part_of_same_object_as": seed["en"]}
    _gi_add_metric_for_kind(kind="island", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    ru_parts = [f"входящих в тот же более крупный географический объект, что и «{seed['ru']}»"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"part of the same larger geographic feature as \"{seed['en']}\""] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("island", requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_island_same_part_of_area",
        template_family="island_part_of", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(bridge="part_of", constraint_key="part_of_same_object_as", property_id="P361", seed=seed),
    )


def _gi_tpl_river_same_basin_country(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    seed = _gi_seed("river", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 4 if complexity == "L3" else 3
    where = [
        _gi_type_line("river"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P205 ?bridgeBasinCountry .",
        "?item wdt:P205 ?bridgeBasinCountry .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "river", "basin_country_from_object": seed["en"]}
    _gi_add_metric_for_kind(kind="river", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    if complexity == "L5" and rng.random() < 0.5:
        cont = rng.choice(CONTINENTS)
        where.append(_gi_continent_line(cont))
        constraints["continent"] = cont["en"]
        geo_ru = f", расположенных {_gi_continent_phrase_ru(cont)}"
        geo_en = f", located in {cont['en']}"
    else:
        geo_ru = ""
        geo_en = ""
    ru_parts = [f"имеющих одну из тех же стран водосборного бассейна, что и «{seed['ru']}»{geo_ru}"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"sharing one of the same basin countries as \"{seed['en']}\"{geo_en}"] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("river", requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_river_same_basin_country_length",
        template_family="river_basin_country", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?length_km"],
        bridge_meta=_gi_bridge_meta(bridge="basin_country", constraint_key="basin_country_from_object", property_id="P205", seed=seed),
    )


def _gi_tpl_lake_same_basin_country_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    seed = _gi_seed("lake", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 4 if complexity == "L3" else 3
    where = [
        _gi_type_line("lake"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P205 ?bridgeBasinCountry .",
        "?item wdt:P205 ?bridgeBasinCountry .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "lake", "basin_country_from_object": seed["en"]}
    _gi_add_metric_for_kind(kind="lake", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    ru_parts = [f"имеющих одну из тех же стран водосборного бассейна, что и «{seed['ru']}»"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"sharing one of the same basin countries as \"{seed['en']}\""] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("lake", requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_lake_same_basin_country_area",
        template_family="lake_basin_country", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(bridge="basin_country", constraint_key="basin_country_from_object", property_id="P205", seed=seed),
    )


def _gi_tpl_lake_same_outflow_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    seed = _gi_seed("lake", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 3
    where = [
        _gi_type_line("lake"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P201 ?bridgeOutflow .",
        "?item wdt:P201 ?bridgeOutflow .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "lake", "outflow_from_object": seed["en"]}
    _gi_add_metric_for_kind(kind="lake", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    ru_parts = [f"имеющих тот же сток, что и «{seed['ru']}»"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"with the same outflow as \"{seed['en']}\""] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("lake", requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_lake_same_outflow_area",
        template_family="lake_outflow", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(bridge="outflow", constraint_key="outflow_from_object", property_id="P201", seed=seed),
    )


def _gi_tpl_cross_kind_same_country_from_seed(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Harder cross-kind bridge: answer kind differs from seed kind, country is hidden."""
    if complexity not in {"L4", "L5"}:
        return None
    seed = rng.choice([s for s in GEO_SEEDS if s["kind"] in {"mountain", "volcano", "lake", "island", "waterfall", "desert"}])
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None
    possible = ["mountain", "volcano", "river", "lake", "waterfall", "island", "desert"]
    possible = [k for k in possible if k != seed["kind"]]
    kind = rng.choice(possible)
    requested = 3
    where = [
        _gi_type_line(kind),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P17 ?bridgeCountry .",
        "?item wdt:P17 ?bridgeCountry .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": kind, "country_from_object": seed["en"], "answer_kind_differs_from_seed": True}
    _gi_add_metric_for_kind(kind=kind, complexity=complexity, rng=rng, where=where, constraints=constraints, force=(complexity == "L5"), prefer_range=(complexity == "L5"))
    ru_parts = [f"расположенных в одной из стран, где находится «{seed['ru']}»"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"located in one of the countries of \"{seed['en']}\""] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_cross_kind_same_country_metric",
        template_family="cross_kind_same_country", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
        bridge_meta=_gi_bridge_meta(bridge="country", constraint_key="country_from_object", property_id="P17", seed=seed, extra={"cross_kind": True}),
    )


def _gi_tpl_same_continent_two_constraints(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    seed = rng.choice([s for s in GEO_SEEDS if s["kind"] in {"desert", "sea", "lake", "island", "mountain", "river"}])
    seed_qid = _gi_seed_qid(seed)
    if not seed_qid:
        return None
    kind = _gi_pick_weighted(rng, ["mountain", "river", "lake", "island", "desert", "sea"], [0.22, 0.22, 0.18, 0.18, 0.10, 0.10])
    requested = 3
    where = [
        _gi_type_line(kind),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P30 ?bridgeContinent .",
        "?item wdt:P30 ?bridgeContinent .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": kind, "continent_from_object": seed["en"]}
    _gi_add_metric_for_kind(kind=kind, complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    # Add country only sometimes in L5 to create 2 explicit constraints plus hidden bridge.
    if complexity == "L5" and rng.random() < 0.45:
        country = rng.choice(COUNTRIES)
        where.append(_gi_country_line(country))
        constraints["country"] = country["en"]
        ru_extra = f", расположенных {_gi_country_phrase_ru(country)}"
        en_extra = f", located in {country['en']}"
    else:
        ru_extra = ""
        en_extra = ""
    ru_parts = [f"находящихся на том же континенте, что и «{seed['ru']}»{ru_extra}"] + _gi_metric_phrase_ru(constraints)
    en_parts = [f"located on the same continent as \"{seed['en']}\"{en_extra}"] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text(kind, requested, ru_parts, en_parts)
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_same_continent_two_constraints",
        template_family="same_continent_hard", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?elevation_m", "?area_sqkm", "?length_km"],
        bridge_meta=_gi_bridge_meta(bridge="continent", constraint_key="continent_from_object", property_id="P30", seed=seed),
    )


def _gi_tpl_mountain_same_range_country_elevation(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    seed = _gi_seed("mountain", rng)
    if not seed:
        return None
    seed_qid = _gi_seed_qid(seed)
    requested = 3
    where = [
        _gi_type_line("mountain"),
        f"BIND(wd:{seed_qid} AS ?seed) .",
        "?seed wdt:P4552 ?bridgeRange .",
        "?item wdt:P4552 ?bridgeRange .",
        "FILTER(?item != ?seed) .",
    ]
    constraints = {"kind": "mountain", "mountain_range_from_object": seed["en"]}
    if complexity == "L5":
        # Add same country as seed as a second hidden bridge only for L5.
        where += ["?seed wdt:P17 ?bridgeCountry .", "?item wdt:P17 ?bridgeCountry ."]
        constraints["country_from_object"] = seed["en"]
    _gi_add_metric_for_kind(kind="mountain", complexity=complexity, rng=rng, where=where, constraints=constraints, force=True, prefer_range=(complexity == "L5"))
    ru_base = f"из того же горного хребта, что и «{seed['ru']}»"
    en_base = f"from the same mountain range as \"{seed['en']}\""
    if complexity == "L5":
        ru_base += f" и в одной из стран, где находится «{seed['ru']}»"
        en_base += f" and in one of the countries of \"{seed['en']}\""
    ru_parts = [ru_base] + _gi_metric_phrase_ru(constraints)
    en_parts = [en_base] + _gi_metric_phrase_en(constraints)
    q_ru, q_en = _gi_finalize_text("mountain", requested, ru_parts, en_parts)
    bridge_meta = _gi_bridge_meta(
        bridge="mountain_range_and_country" if complexity == "L5" else "mountain_range",
        constraint_key="mountain_range_from_object",
        property_chain=["P4552"] + (["P17"] if complexity == "L5" else []),
        seed=seed,
    )
    return _gi_finalize_wdqs_example(
        idx=idx, complexity=complexity, template_id="geo_int_mountain_range_country_elevation",
        template_family="mountain_range_hard", q_ru=q_ru, q_en=q_en, constraints=constraints,
        where_lines=where, requested_count=requested, extra_select_vars=["?elevation_m"], bridge_meta=bridge_meta,
    )


# -------------------------
# Weighted template registry
# -------------------------

GEO_INT_TEMPLATE_REGISTRY = {
    "L1": [
        (_gi_tpl_country_kind, 1.0),
        (_gi_tpl_continent_kind, 1.0),
    ],
    "L2": [
        (_gi_tpl_country_kind, 1.0),
        (_gi_tpl_continent_kind, 1.0),
    ],

    # L3 is no longer allowed to use plain one-hop country_kind / continent_kind.
    # Starting from L3, every record must be hidden-bridge or strong multi-constraint.
    "L3": [
        (_gi_tpl_mountain_range_elevation, 1.4),
        (_gi_tpl_same_country_metric, 1.2),
        (_gi_tpl_same_continent_as_seed, 1.1),
        (_gi_tpl_country_continent_metric, 0.8),
        (_gi_tpl_island_same_water_body_area, 1.0),
        (_gi_tpl_river_same_mouth, 1.1),
        (_gi_tpl_river_same_basin_country, 1.0),
        (_gi_tpl_lake_same_basin_country_area, 0.9),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.0),
    ],

    "L4": [
        (_gi_tpl_mountain_range_elevation, 1.0),
        (_gi_tpl_mountain_same_range_country_elevation, 1.1),
        (_gi_tpl_same_country_metric, 0.9),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.2),
        (_gi_tpl_river_same_mouth, 1.2),
        (_gi_tpl_river_same_basin_country, 1.2),
        (_gi_tpl_lake_same_basin_country_area, 1.0),
        (_gi_tpl_lake_same_outflow_area, 0.9),
        (_gi_tpl_island_same_water_body_area, 1.0),
        (_gi_tpl_island_same_part_of_area, 0.8),
        (_gi_tpl_same_continent_as_seed, 0.8),
        (_gi_tpl_same_continent_two_constraints, 1.0),
        (_gi_tpl_country_continent_metric, 0.7),
    ],

    "L5": [
        (_gi_tpl_mountain_range_elevation, 1.0),
        (_gi_tpl_mountain_same_range_country_elevation, 1.2),
        (_gi_tpl_same_country_metric, 0.8),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.3),
        (_gi_tpl_river_same_mouth, 1.2),
        (_gi_tpl_river_same_basin_country, 1.3),
        (_gi_tpl_lake_same_basin_country_area, 1.0),
        (_gi_tpl_lake_same_outflow_area, 1.0),
        (_gi_tpl_island_same_water_body_area, 1.0),
        (_gi_tpl_island_same_part_of_area, 0.9),
        (_gi_tpl_same_continent_two_constraints, 1.1),
        (_gi_tpl_country_continent_metric, 0.7),
    ],
}

# Backward list form for audit/debug compatibility.
GEO_INT_TEMPLATE_FNS = {lvl: [fn for fn, _w in entries] for lvl, entries in GEO_INT_TEMPLATE_REGISTRY.items()}


def _gi_weighted_template_order(complexity: str, idx: int, rng: random.Random) -> List[Any]:
    entries = list(GEO_INT_TEMPLATE_REGISTRY.get(complexity, []))
    if not entries:
        return []
    fns = [fn for fn, _w in entries]
    weights = [float(w) for _fn, w in entries]
    primary = _gi_pick_weighted(rng, fns, weights)
    rest = [fn for fn in fns if fn is not primary]
    rng.shuffle(rest)
    return [primary] + rest


def generate_geo_international_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 100,
) -> BenchmarkExample:
    if complexity not in GEO_INT_TEMPLATE_REGISTRY:
        raise ValueError(f"Unknown complexity: {complexity}")

    last_error = None
    for attempt in range(max_attempts):
        ordered = _gi_weighted_template_order(complexity, idx + attempt, rng)
        for fn in ordered:
            try:
                ex = fn(complexity, idx, rng)
                if ex is not None and getattr(ex, "gold_answer_qids", None):
                    return ex
            except Exception as e:
                last_error = f"{type(e).__name__}: {str(e)[:300]}"

    return BenchmarkExample(
        id=f"geo_int_fail_{complexity.lower()}_{idx:04d}",
        domain="geo_international",
        complexity=complexity,
        query_text_ru="(ошибка генерации geo_international)",
        query_text_en="(geo_international generation failed)",
        constraints={"failed": True, "complexity": complexity, "last_error": last_error},
        requested_count=3,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        gold_answer_labels_en=[],
        sparql_query="",
        created_at=utc_now_z(),
        template_id="geo_int_failed",
        template_family="failed",
        local_validator=None,
        gold_collection_meta={"failed": True, "last_error": last_error},
    )


def generate_geo_world_example(complexity: str, idx: int, rng: random.Random, max_attempts: int = 100) -> BenchmarkExample:
    return generate_geo_international_example(complexity, idx, rng, max_attempts=max_attempts)

if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["geo"] = generate_geo_world_example
    DOMAIN_GENERATORS["geo_international"] = generate_geo_international_example

print("geo_international hard-template patch loaded: diverse weighted L3-L5 multihop patterns")


geo_international hard-template patch loaded: diverse weighted L3-L5 multihop patterns


## Final quality patch: richer L3–L5 mixed-geography patterns

In [46]:

# ============================================================
# Final quality patch: richer L3-L5 mixed-geography patterns
# ============================================================
# This cell is intentionally placed after the base hard-template patch and before
# generation. It overrides the hard-level registry one more time:
# - L3 has no simple one-hop country/continent templates.
# - basin-country templates are downweighted so L3 does not repeat the same pattern.
# - tautological country->continent template is removed from L3-L5.
# - mixed-geo templates are added, e.g. rivers in countries that also contain large deserts.


def _gi_related_desert_area_lines(min_sqkm: int) -> List[str]:
    return [
        "?bridgeDesert wdt:P31/wdt:P279* wd:Q8514 .",
        "?bridgeDesert wdt:P17 ?bridgeCountry .",
    ] + _gi_metric_value_lines("bridgeDesert", "area_sqkm", "bridge_desert_area_sqkm", "bridgeDesertArea") + [
        f"FILTER(?bridge_desert_area_sqkm >= {int(min_sqkm)}) .",
    ]


def _gi_related_mountain_elevation_lines(min_m: int) -> List[str]:
    return [
        "?bridgeMountain wdt:P31/wdt:P279* wd:Q8502 .",
        "?bridgeMountain wdt:P17 ?bridgeCountry .",
    ] + _gi_metric_value_lines("bridgeMountain", "elevation_m", "bridge_mountain_elevation_m", "bridgeMountainElevation") + [
        f"FILTER(?bridge_mountain_elevation_m >= {int(min_m)}) .",
    ]


def _gi_related_volcano_elevation_lines(min_m: int) -> List[str]:
    return [
        "?bridgeVolcano wdt:P31/wdt:P279* wd:Q8072 .",
        "?bridgeVolcano wdt:P17 ?bridgeCountry .",
    ] + _gi_metric_value_lines("bridgeVolcano", "elevation_m", "bridge_volcano_elevation_m", "bridgeVolcanoElevation") + [
        f"FILTER(?bridge_volcano_elevation_m >= {int(min_m)}) .",
    ]


def _gi_related_river_length_lines(min_km: int) -> List[str]:
    return [
        "?bridgeRiver wdt:P31 wd:Q4022 .",
        "?bridgeRiver wdt:P17 ?bridgeCountry .",
    ] + _gi_metric_value_lines("bridgeRiver", "length_km", "bridge_river_length_km", "bridgeRiverLength") + [
        f"FILTER(?bridge_river_length_km >= {int(min_km)}) .",
    ]


def _gi_tpl_rivers_in_countries_with_large_deserts(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Rivers with a length constraint, located in a country that also contains a large desert."""
    if complexity not in {"L3", "L4", "L5"}:
        return None

    requested = 4 if complexity == "L3" else 3
    desert_min = rng.choice([500, 1000, 5000, 10000]) if complexity != "L5" else rng.choice([1000, 5000, 10000, 50000])

    where = [
        _gi_type_line("river"),
        "?item wdt:P17 ?bridgeCountry .",
    ]
    where += _gi_related_desert_area_lines(desert_min)

    constraints = {
        "kind": "river",
        "country_contains_kind": "desert",
        "related_desert_area_min_sqkm": desert_min,
    }

    if complexity == "L3":
        max_len = rng.choice([100, 250, 500, 1000])
        where += _gi_length_max_lines(max_len)
        constraints["length_max_km"] = max_len
    elif complexity == "L4":
        if rng.random() < 0.55:
            max_len = rng.choice([100, 250, 500, 1000])
            where += _gi_length_max_lines(max_len)
            constraints["length_max_km"] = max_len
        else:
            min_len = rng.choice([250, 500, 1000, 2000])
            where += _gi_length_min_lines(min_len)
            constraints["length_min_km"] = min_len
    else:
        lo = rng.choice([100, 250, 500])
        hi = lo + rng.choice([250, 500, 1000])
        where += _gi_length_range_lines(lo, hi)
        constraints["length_min_km"] = lo
        constraints["length_max_km"] = hi

    if constraints.get("length_max_km") is not None and constraints.get("length_min_km") is None:
        ru_metric = f"с длиной не более {constraints['length_max_km']} км"
        en_metric = f"with length at most {constraints['length_max_km']} km"
    else:
        ru_metric = _gi_metric_phrase_ru(constraints)[0]
        en_metric = _gi_metric_phrase_en(constraints)[0]

    ru_parts = [
        ru_metric,
        f"расположенных в стране, где есть пустыня площадью не менее {desert_min} км²",
    ]
    en_parts = [
        en_metric,
        f"located in a country that contains a desert with area at least {desert_min} sq km",
    ]
    q_ru, q_en = _gi_finalize_text("river", requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_river_country_with_large_desert",
        template_family="mixed_country_desert_river",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?length_km"],
        bridge_meta=_gi_bridge_meta(
            bridge="country_contains_desert",
            constraint_key="country_contains_kind",
            property_chain=["P17", "P2046"],
            semantics="answer and related desert share a country; related desert is filtered by area",
            extra={"related_kind": "desert", "related_metric": "area_min_sqkm", "related_metric_value": desert_min},
        ),
    )


def _gi_tpl_lakes_in_countries_with_high_mountains(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Lakes with an area constraint, located in a country that also contains high mountains."""
    if complexity not in {"L3", "L4", "L5"}:
        return None

    requested = 4 if complexity == "L3" else 3
    mountain_min = rng.choice([1500, 2000, 3000, 4000]) if complexity != "L5" else rng.choice([3000, 4000, 5000, 6000])

    where = [
        _gi_type_line("lake"),
        "?item wdt:P17 ?bridgeCountry .",
    ]
    where += _gi_related_mountain_elevation_lines(mountain_min)

    constraints = {
        "kind": "lake",
        "country_contains_kind": "mountain",
        "related_mountain_elevation_min_m": mountain_min,
    }

    if complexity == "L5":
        lo = rng.choice([100, 500, 1000])
        hi = lo * rng.choice([2, 3, 5])
        where += _gi_area_range_lines(lo, hi)
        constraints["area_min_sqkm"] = lo
        constraints["area_max_sqkm"] = hi
    elif rng.random() < 0.45:
        max_area = rng.choice([50, 100, 500, 1000])
        where += _gi_area_max_lines(max_area)
        constraints["area_max_sqkm"] = max_area
    else:
        min_area = rng.choice([100, 500, 1000, 5000])
        where += _gi_area_min_lines(min_area)
        constraints["area_min_sqkm"] = min_area

    ru_parts = _gi_metric_phrase_ru(constraints) + [
        f"расположенных в стране, где есть гора высотой не ниже {mountain_min} м",
    ]
    en_parts = _gi_metric_phrase_en(constraints) + [
        f"located in a country that contains a mountain with elevation at least {mountain_min} m",
    ]
    q_ru, q_en = _gi_finalize_text("lake", requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_lake_country_with_high_mountain",
        template_family="mixed_country_mountain_lake",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(
            bridge="country_contains_mountain",
            constraint_key="country_contains_kind",
            property_chain=["P17", "P2044"],
            semantics="answer and related mountain share a country; related mountain is filtered by elevation",
            extra={"related_kind": "mountain", "related_metric": "elevation_min_m", "related_metric_value": mountain_min},
        ),
    )


def _gi_tpl_waterfalls_in_countries_with_high_volcanoes(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Waterfalls located in countries that also contain high volcanoes."""
    if complexity not in {"L4", "L5"}:
        return None

    requested = 3
    volcano_min = rng.choice([1000, 1500, 2000, 3000]) if complexity == "L4" else rng.choice([2000, 3000, 4000, 5000])

    where = [
        _gi_type_line("waterfall"),
        "?item wdt:P17 ?bridgeCountry .",
    ]
    where += _gi_related_volcano_elevation_lines(volcano_min)

    constraints = {
        "kind": "waterfall",
        "country_contains_kind": "volcano",
        "related_volcano_elevation_min_m": volcano_min,
    }

    # Add a continent filter in L5 sometimes to make it more selective and less broad.
    if complexity == "L5" and rng.random() < 0.50:
        cont = rng.choice(CONTINENTS)
        where.append(f"?item wdt:P30 wd:{cont['qid']} .")
        constraints["continent"] = cont["en"]
        geo_ru = f", расположенных {_gi_continent_phrase_ru(cont)}"
        geo_en = f", located in {cont['en']}"
    else:
        geo_ru = ""
        geo_en = ""

    ru_parts = [f"расположенных в стране, где есть вулкан высотой не ниже {volcano_min} м{geo_ru}"]
    en_parts = [f"located in a country that contains a volcano with elevation at least {volcano_min} m{geo_en}"]
    q_ru, q_en = _gi_finalize_text("waterfall", requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_waterfall_country_with_high_volcano",
        template_family="mixed_country_volcano_waterfall",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        bridge_meta=_gi_bridge_meta(
            bridge="country_contains_volcano",
            constraint_key="country_contains_kind",
            property_chain=["P17", "P2044"],
            semantics="answer and related volcano share a country; related volcano is filtered by elevation",
            extra={"related_kind": "volcano", "related_metric": "elevation_min_m", "related_metric_value": volcano_min},
        ),
    )


def _gi_tpl_islands_in_countries_with_long_rivers(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    """Islands with area constraints, located in countries that also contain long rivers."""
    if complexity not in {"L4", "L5"}:
        return None

    requested = 3
    river_min = rng.choice([500, 1000, 2000]) if complexity == "L4" else rng.choice([1000, 2000, 3000, 5000])

    where = [
        _gi_type_line("island"),
        "?item wdt:P17 ?bridgeCountry .",
    ]
    where += _gi_related_river_length_lines(river_min)

    constraints = {
        "kind": "island",
        "country_contains_kind": "river",
        "related_river_length_min_km": river_min,
    }

    if complexity == "L5":
        lo = rng.choice([100, 500, 1000])
        hi = lo * rng.choice([2, 5, 10])
        where += _gi_area_range_lines(lo, hi)
        constraints["area_min_sqkm"] = lo
        constraints["area_max_sqkm"] = hi
    elif rng.random() < 0.50:
        min_area = rng.choice([100, 500, 1000])
        where += _gi_area_min_lines(min_area)
        constraints["area_min_sqkm"] = min_area
    else:
        max_area = rng.choice([50, 100, 500, 1000])
        where += _gi_area_max_lines(max_area)
        constraints["area_max_sqkm"] = max_area

    ru_parts = _gi_metric_phrase_ru(constraints) + [
        f"расположенных в стране, где есть река длиной не менее {river_min} км",
    ]
    en_parts = _gi_metric_phrase_en(constraints) + [
        f"located in a country that contains a river with length at least {river_min} km",
    ]
    q_ru, q_en = _gi_finalize_text("island", requested, ru_parts, en_parts)

    return _gi_finalize_wdqs_example(
        idx=idx,
        complexity=complexity,
        template_id="geo_int_island_country_with_long_river",
        template_family="mixed_country_river_island",
        q_ru=q_ru,
        q_en=q_en,
        constraints=constraints,
        where_lines=where,
        requested_count=requested,
        extra_select_vars=["?area_sqkm"],
        bridge_meta=_gi_bridge_meta(
            bridge="country_contains_river",
            constraint_key="country_contains_kind",
            property_chain=["P17", "P2043"],
            semantics="answer and related river share a country; related river is filtered by length",
            extra={"related_kind": "river", "related_metric": "length_min_km", "related_metric_value": river_min},
        ),
    )


# Final weighted template registry.
# L3 has no simple one-hop templates and the most repetitive basin-country pattern is downweighted.
# L4/L5 include mixed-geo country relation templates.
GEO_INT_TEMPLATE_REGISTRY = {
    "L1": [
        (_gi_tpl_country_kind, 1.0),
        (_gi_tpl_continent_kind, 1.0),
    ],
    "L2": [
        (_gi_tpl_country_kind, 1.0),
        (_gi_tpl_continent_kind, 1.0),
    ],

    "L3": [
        (_gi_tpl_mountain_range_elevation, 1.5),
        (_gi_tpl_same_country_metric, 0.9),
        (_gi_tpl_same_continent_as_seed, 0.6),
        (_gi_tpl_island_same_water_body_area, 1.2),
        (_gi_tpl_river_same_mouth, 1.3),
        (_gi_tpl_river_same_basin_country, 0.45),
        (_gi_tpl_lake_same_basin_country_area, 0.45),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.2),
        (_gi_tpl_rivers_in_countries_with_large_deserts, 1.2),
        (_gi_tpl_lakes_in_countries_with_high_mountains, 1.0),
    ],

    "L4": [
        (_gi_tpl_mountain_range_elevation, 1.0),
        (_gi_tpl_mountain_same_range_country_elevation, 1.2),
        (_gi_tpl_same_country_metric, 0.7),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.1),
        (_gi_tpl_river_same_mouth, 1.2),
        (_gi_tpl_river_same_basin_country, 0.75),
        (_gi_tpl_lake_same_basin_country_area, 0.75),
        (_gi_tpl_lake_same_outflow_area, 1.0),
        (_gi_tpl_island_same_water_body_area, 1.1),
        (_gi_tpl_island_same_part_of_area, 0.8),
        (_gi_tpl_same_continent_two_constraints, 0.9),
        (_gi_tpl_rivers_in_countries_with_large_deserts, 1.2),
        (_gi_tpl_lakes_in_countries_with_high_mountains, 1.0),
        (_gi_tpl_waterfalls_in_countries_with_high_volcanoes, 0.9),
        (_gi_tpl_islands_in_countries_with_long_rivers, 0.9),
    ],

    "L5": [
        (_gi_tpl_mountain_range_elevation, 0.9),
        (_gi_tpl_mountain_same_range_country_elevation, 1.25),
        (_gi_tpl_same_country_metric, 0.6),
        (_gi_tpl_cross_kind_same_country_from_seed, 1.1),
        (_gi_tpl_river_same_mouth, 1.1),
        (_gi_tpl_river_same_basin_country, 0.75),
        (_gi_tpl_lake_same_basin_country_area, 0.75),
        (_gi_tpl_lake_same_outflow_area, 1.1),
        (_gi_tpl_island_same_water_body_area, 1.0),
        (_gi_tpl_island_same_part_of_area, 0.8),
        (_gi_tpl_same_continent_two_constraints, 1.0),
        (_gi_tpl_rivers_in_countries_with_large_deserts, 1.25),
        (_gi_tpl_lakes_in_countries_with_high_mountains, 1.1),
        (_gi_tpl_waterfalls_in_countries_with_high_volcanoes, 1.0),
        (_gi_tpl_islands_in_countries_with_long_rivers, 1.0),
    ],
}

GEO_INT_TEMPLATE_FNS = {lvl: [fn for fn, _w in entries] for lvl, entries in GEO_INT_TEMPLATE_REGISTRY.items()}

print("geo_international final quality patch loaded: no one-hop L3, L2 min/max metrics, diversified L3-L5, mixed-geo templates")


geo_international final quality patch loaded: no one-hop L3, L2 min/max metrics, diversified L3-L5, mixed-geo templates


## Final numeric-unit and quality patch

This version normalizes Wikidata quantity units for area/length/elevation, makes L2 always include a metric, makes L3+ require a numeric or related metric, and applies stronger lake/island/river quality filtering.

## Run geo_international generation

Generates the domain output incrementally with progress bars, resume support, audit, and checkpoint files.


In [47]:
# Legacy WDQS-per-example generation is intentionally disabled.
# Use the cache-first generation block below instead.
RUN_GEO_INT_LEGACY_WDQS_GENERATION = False
print("Legacy WDQS-per-example geo generation is disabled. Use cache-first v3 cells below.")

Legacy WDQS-per-example geo generation is disabled. Use cache-first v3 cells below.


## Cache-first geo_international v3

Builds a broad Wikidata-derived local cache once, then generates L1–L5 examples locally from that cache.

In [48]:
# ============================================================
# Cache-first geo_international v3 — fixed hard generation
# ============================================================
# Goal:
# - Build a wide Wikidata-derived local geo cache once.
# - Generate diverse L3-L5 examples locally from cache.
# - Keep full gold lists for each generated query inside the cache.
# - Preserve the cinema-style output schema: RU/EN queries, clean constraints,
#   ask_validator_sparql, local_validator, gold_collection_meta, bridge_meta.

from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import json
import random
import re
import time
import requests

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

CACHE_DIR_GEO_V3 = Path("out_wikidata_benchmark/cache")
CACHE_DIR_GEO_V3.mkdir(parents=True, exist_ok=True)
DOMAIN_OUT_DIR_GEO_V3 = Path("out_wikidata_benchmark/domain_outputs")
DOMAIN_OUT_DIR_GEO_V3.mkdir(parents=True, exist_ok=True)

GEO_CACHE_V3_PATH = CACHE_DIR_GEO_V3 / "geo_international_cache_v3.json"
GEO_CACHE_V3_AUDIT_PATH = CACHE_DIR_GEO_V3 / "geo_international_cache_v3_audit.json"
GEO_CACHE_OUT_PATH = DOMAIN_OUT_DIR_GEO_V3 / "geo_international.jsonl"
GEO_CACHE_AUDIT_PATH = DOMAIN_OUT_DIR_GEO_V3 / "geo_international_generation_audit.json"
GEO_CACHE_CHECKPOINT_PATH = DOMAIN_OUT_DIR_GEO_V3 / "geo_international_generation_checkpoint.json"

REBUILD_GEO_CACHE_V3 = False
RUN_GEO_CACHE_FIRST_GENERATION = True

TARGET_PLAN_GEO_CACHE = {"L1": 0, "L2": 0, "L3": 0, "L4": 25, "L5": 35}
GEO_CACHE_SEED = 20260521
GEO_CACHE_RNG = random.Random(GEO_CACHE_SEED)
REQUIRE_CACHE_COMPLETE_FOR_GOLD = False  # important: global kind caches may hit limits; do not reject all hard examples

GEO_CACHE_MAX_GOLD_BY_LEVEL = {"L1": 80, "L2": 70, "L3": 50, "L4": 65, "L5": 60}
GEO_CACHE_MIN_GOLD_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}
GEO_CACHE_REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}
GEO_CACHE_MAX_ATTEMPTS_BY_LEVEL = {"L1": 600, "L2": 800, "L3": 1200, "L4": 2500, "L5": 3500}

WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
WDQS_HEADERS = {"Accept": "application/sparql-results+json", "User-Agent": "multihop-benchmark-geo-cache-v3/1.0"}

GEO_KIND_CFG = {
    "mountain": {"qid": "Q8502", "property": "P2044", "metric": "elevation_m", "ru_acc": "гор", "en_plural": "mountains", "direct_only": False, "cache_limit": 14000, "quality_blacklist": []},
    "volcano": {"qid": "Q8072", "property": "P2044", "metric": "elevation_m", "ru_acc": "вулканов", "en_plural": "volcanoes", "direct_only": False, "cache_limit": 9000, "quality_blacklist": []},
    "river": {"qid": "Q4022", "property": "P2043", "metric": "length_km", "ru_acc": "рек", "en_plural": "rivers", "direct_only": True, "cache_limit": 16000, "quality_blacklist": [r"\bcreek\b", r"\bstream\b", r"\bbrook\b", r"\branch\b", r"\bcanal\b", r"\bditch\b"]},
    "lake": {"qid": "Q23397", "property": "P2046", "metric": "area_sqkm", "ru_acc": "озёр", "en_plural": "lakes", "direct_only": True, "cache_limit": 12000, "quality_blacklist": [r"forest park", r"natural reserve", r"provincial natural reserve", r"reservoir region", r"national forest", r"playa", r"shuiku"]},
    "island": {"qid": "Q23442", "property": "P2046", "metric": "area_sqkm", "ru_acc": "островов", "en_plural": "islands", "direct_only": True, "cache_limit": 16000, "quality_blacklist": [r"\breef(s)?\b", r"\bbank(s)?\b", r"\brock(s)?\b", r"\bshoal(s)?\b", r"\bsandbank(s)?\b"]},
    "desert": {"qid": "Q8514", "property": "P2046", "metric": "area_sqkm", "ru_acc": "пустынь", "en_plural": "deserts", "direct_only": False, "cache_limit": 6000, "quality_blacklist": []},
    "waterfall": {"qid": "Q34038", "property": None, "metric": None, "ru_acc": "водопадов", "en_plural": "waterfalls", "direct_only": False, "cache_limit": 10000, "quality_blacklist": []},
    "sea": {"qid": "Q165", "property": "P2046", "metric": "area_sqkm", "ru_acc": "морей", "en_plural": "seas", "direct_only": False, "cache_limit": 2500, "quality_blacklist": []},
}
FALLBACK_QID_LABELS_EN = {"Q15": "Africa", "Q46": "Europe", "Q48": "Asia", "Q18": "South America", "Q49": "North America", "Q51": "Antarctica", "Q538": "Oceania"}
FALLBACK_QID_LABELS_RU = {"Q15": "Африке", "Q46": "Европе", "Q48": "Азии", "Q18": "Южной Америке", "Q49": "Северной Америке", "Q51": "Антарктиде", "Q538": "Океании"}


def _qid_from_uri(uri: str | None) -> str | None:
    if not uri:
        return None
    m = re.search(r"/(Q\d+)$", str(uri))
    return m.group(1) if m else None


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _geo_wdqs(query: str, *, retries: int = 3, timeout: int = 120, sleep_s: float = 1.5) -> list[dict]:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(WDQS_ENDPOINT, params={"query": query, "format": "json"}, headers=WDQS_HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp.json().get("results", {}).get("bindings", [])
            last_error = RuntimeError(f"WDQS status {resp.status_code}: {resp.text[:400]}")
        except Exception as e:
            last_error = e
        time.sleep(sleep_s * attempt)
    raise RuntimeError(f"WDQS failed after {retries} retries: {last_error}")


def _quantity_optional_lines(prop: str, var: str, unit_kind: str) -> str:
    amount = f"?{var}Amount"; unit = f"?{var}Unit"; node = f"?{var}Value"; stmt = f"?{var}Statement"
    if unit_kind == "area_sqkm":
        expr = f"IF({unit} = wd:Q712226, xsd:decimal({amount}), IF({unit} = wd:Q25343, xsd:decimal({amount}) / 1000000, IF({unit} = wd:Q35852, xsd:decimal({amount}) / 100, xsd:decimal({amount}))))"
    elif unit_kind == "length_km":
        expr = f"IF({unit} = wd:Q828224, xsd:decimal({amount}), IF({unit} = wd:Q11573, xsd:decimal({amount}) / 1000, IF({unit} = wd:Q3710, xsd:decimal({amount}) * 0.0003048, xsd:decimal({amount}))))"
    elif unit_kind == "elevation_m":
        expr = f"IF({unit} = wd:Q11573, xsd:decimal({amount}), IF({unit} = wd:Q828224, xsd:decimal({amount}) * 1000, IF({unit} = wd:Q3710, xsd:decimal({amount}) * 0.3048, xsd:decimal({amount}))))"
    else:
        expr = f"xsd:decimal({amount})"
    return f"""
      OPTIONAL {{
        ?item p:{prop} {stmt} .
        {stmt} psv:{prop} {node} .
        {node} wikibase:quantityAmount {amount} .
        {node} wikibase:quantityUnit {unit} .
        BIND(({expr}) AS ?{unit_kind}) .
      }}"""


def _quantity_required_filter_lines(prop: str, var_name: str, metric: str, op: str, value) -> str:
    optional = _quantity_optional_lines(prop, var_name, metric)
    required = optional.replace("OPTIONAL {", "{")
    return required + f"\n      FILTER(?{metric} {op} {value}) ."


def _kind_type_line(kind: str) -> str:
    cfg = GEO_KIND_CFG[kind]
    if cfg["direct_only"]:
        return f"?item wdt:P31 wd:{cfg['qid']} ."
    return f"?item wdt:P31/wdt:P279* wd:{cfg['qid']} ."


def _build_kind_cache_query(kind: str) -> str:
    cfg = GEO_KIND_CFG[kind]
    metric = cfg.get("metric")
    metric_select = f"?{metric}" if metric else ""
    metric_lines = _quantity_optional_lines(cfg["property"], "itemMetric", metric) if cfg.get("property") and metric else ""
    bridge_lines = ""
    if kind == "mountain":
        bridge_lines += "\n      OPTIONAL { ?item wdt:P4552 ?mountain_range . }"
    if kind == "river":
        bridge_lines += "\n      OPTIONAL { ?item wdt:P205 ?basin_country . }\n      OPTIONAL { ?item wdt:P403 ?mouth_body . }"
    if kind == "lake":
        bridge_lines += "\n      OPTIONAL { ?item wdt:P205 ?basin_country . }\n      OPTIONAL { ?item wdt:P201 ?outflow . }"
    if kind == "island":
        bridge_lines += "\n      OPTIONAL { ?item wdt:P206 ?water_body . }\n      OPTIONAL { ?item wdt:P361 ?part_of . }"
    return f"""
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu ?country ?continent {metric_select}
       ?mountain_range ?basin_country ?mouth_body ?outflow ?water_body ?part_of
WHERE {{
      {_kind_type_line(kind)}
      OPTIONAL {{ ?item wdt:P17 ?country . }}
      OPTIONAL {{ ?item wdt:P30 ?directContinent . }}
      OPTIONAL {{ ?country wdt:P30 ?countryContinent . }}
      BIND(COALESCE(?directContinent, ?countryContinent) AS ?continent) .
      {metric_lines}
      {bridge_lines}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
}}
LIMIT {cfg['cache_limit']}
"""


def _row_value(row: dict, key: str):
    return row.get(key, {}).get("value")


def _as_float(x):
    try:
        return float(x) if x is not None else None
    except Exception:
        return None


def _label_is_noisy(kind: str, label: str) -> bool:
    lower = (label or "").lower()
    return any(re.search(p, lower) for p in GEO_KIND_CFG[kind].get("quality_blacklist", []))


def _rows_to_entities(kind: str, rows: list[dict]) -> tuple[list[dict], dict]:
    grouped = {}; dropped_no_qid = 0; dropped_no_label = 0; dropped_noise = []
    for row in rows:
        qid = _qid_from_uri(_row_value(row, "item")); label_en = _row_value(row, "itemLabelEn")
        if not qid:
            dropped_no_qid += 1; continue
        if not label_en:
            dropped_no_label += 1; continue
        if _label_is_noisy(kind, label_en):
            dropped_noise.append({"qid": qid, "label_en": label_en}); continue
        ent = grouped.setdefault(qid, {"qid": qid, "kind": kind, "label_en": label_en, "label_ru": _row_value(row, "itemLabelRu") or label_en, "country_qids": set(), "continent_qids": set(), "metric": {}, "mountain_range_qids": set(), "basin_country_qids": set(), "mouth_body_qids": set(), "outflow_qids": set(), "water_body_qids": set(), "part_of_qids": set()})
        for src, dst in [("country", "country_qids"), ("continent", "continent_qids"), ("mountain_range", "mountain_range_qids"), ("basin_country", "basin_country_qids"), ("mouth_body", "mouth_body_qids"), ("outflow", "outflow_qids"), ("water_body", "water_body_qids"), ("part_of", "part_of_qids")]:
            q = _qid_from_uri(_row_value(row, src))
            if q: ent[dst].add(q)
        metric = GEO_KIND_CFG[kind].get("metric")
        if metric and row.get(metric):
            val = _as_float(_row_value(row, metric))
            if val is not None:
                old = ent["metric"].get(metric)
                ent["metric"][metric] = val if old is None else max(old, val)
    entities = []
    for ent in grouped.values():
        for k in list(ent.keys()):
            if k.endswith("_qids"):
                ent[k] = sorted(ent[k])
        entities.append(ent)
    audit = {"raw_rows": len(rows), "entities": len(entities), "dropped_no_qid": dropped_no_qid, "dropped_no_en_label": dropped_no_label, "quality_filter_dropped_count": len(dropped_noise), "quality_filter_dropped_preview": dropped_noise[:50]}
    return entities, audit


def build_geo_cache_v3(*, rebuild: bool = False) -> dict:
    if GEO_CACHE_V3_PATH.exists() and not rebuild:
        print("loading existing geo cache:", GEO_CACHE_V3_PATH.resolve())
        with GEO_CACHE_V3_PATH.open("r", encoding="utf-8") as f: return json.load(f)
    cache = {"version": "geo_cache_v3", "created_at": _now_iso(), "source": "Wikidata Query Service", "kinds": {}, "kind_audit": {}, "labels": {"en": dict(FALLBACK_QID_LABELS_EN), "ru": dict(FALLBACK_QID_LABELS_RU)}}
    for kind in GEO_KIND_CFG:
        print(f"[geo cache] querying {kind}...")
        query = _build_kind_cache_query(kind)
        rows = _geo_wdqs(query, retries=3, timeout=180)
        entities, audit = _rows_to_entities(kind, rows)
        limit = GEO_KIND_CFG[kind]["cache_limit"]
        audit.update({"limit": limit, "limit_hit": len(rows) >= limit, "query": query})
        cache["kinds"][kind] = entities; cache["kind_audit"][kind] = audit
        print(f"  {kind}: rows={len(rows)}, entities={len(entities)}, limit_hit={audit['limit_hit']}")
    with GEO_CACHE_V3_PATH.open("w", encoding="utf-8") as f: json.dump(cache, f, ensure_ascii=False)
    with GEO_CACHE_V3_AUDIT_PATH.open("w", encoding="utf-8") as f: json.dump({"kind_audit": cache["kind_audit"]}, f, ensure_ascii=False, indent=2)
    print("cache saved:", GEO_CACHE_V3_PATH.resolve())
    return cache


def _make_geo_indexes(cache: dict) -> dict:
    by_kind = {k: list(v) for k, v in cache["kinds"].items()}; by_qid = {}
    idx = {"by_kind": by_kind, "by_qid": by_qid, "kind_country": defaultdict(list), "kind_continent": defaultdict(list), "mountain_range": defaultdict(list), "basin_country_kind": defaultdict(list), "mouth_body": defaultdict(list), "outflow": defaultdict(list), "water_body": defaultdict(list), "part_of": defaultdict(list), "country_contains": defaultdict(lambda: defaultdict(list))}
    for kind, ents in by_kind.items():
        for e in ents:
            by_qid[e["qid"]] = e
            for c in e.get("country_qids", []): idx["kind_country"][(kind, c)].append(e); idx["country_contains"][c][kind].append(e)
            for cont in e.get("continent_qids", []): idx["kind_continent"][(kind, cont)].append(e)
            for r in e.get("mountain_range_qids", []): idx["mountain_range"][r].append(e)
            for bc in e.get("basin_country_qids", []): idx["basin_country_kind"][(kind, bc)].append(e)
            for mb in e.get("mouth_body_qids", []): idx["mouth_body"][mb].append(e)
            for of in e.get("outflow_qids", []): idx["outflow"][of].append(e)
            for wb in e.get("water_body_qids", []): idx["water_body"][wb].append(e)
            for po in e.get("part_of_qids", []): idx["part_of"][po].append(e)
    return idx


def _metric_value(e: dict, metric: str): return (e.get("metric") or {}).get(metric)

def _filter_metric(ents: list[dict], metric: str | None = None, *, min_v=None, max_v=None) -> list[dict]:
    if not metric: return list(ents)
    out = []
    for e in ents:
        v = _metric_value(e, metric)
        if v is None: continue
        if min_v is not None and v < min_v: continue
        if max_v is not None and v > max_v: continue
        out.append(e)
    return out


def _dedupe_entities(ents: list[dict]) -> list[dict]:
    seen=set(); out=[]
    for e in ents:
        if e["qid"] in seen: continue
        seen.add(e["qid"]); out.append(e)
    return out


def _label_en(cache: dict, qid: str | None) -> str:
    if not qid: return "unknown"
    ent = cache.get("_indexes", {}).get("by_qid", {}).get(qid)
    return (ent.get("label_en") if ent else None) or cache.get("labels", {}).get("en", {}).get(qid, qid)


def _label_ru(cache: dict, qid: str | None) -> str:
    if not qid: return "unknown"
    ent = cache.get("_indexes", {}).get("by_qid", {}).get(qid)
    return (ent.get("label_ru") if ent else None) or cache.get("labels", {}).get("ru", {}).get(qid, _label_en(cache, qid))


def _choose_seed_with_values(rng, ents, field, *, metric=None):
    cands = [e for e in ents if e.get(field)]
    if metric: cands = [e for e in cands if _metric_value(e, metric) is not None]
    return rng.choice(cands) if cands else None


def _metric_phrase_ru(metric, min_v=None, max_v=None):
    if metric == "elevation_m": return f"с высотой от {int(min_v)} до {int(max_v)} м" if min_v is not None and max_v is not None else (f"с высотой не ниже {int(min_v)} м" if min_v is not None else f"с высотой не выше {int(max_v)} м")
    if metric == "length_km": return f"с длиной от {int(min_v)} до {int(max_v)} км" if min_v is not None and max_v is not None else (f"с длиной не менее {int(min_v)} км" if min_v is not None else f"с длиной не более {int(max_v)} км")
    if metric == "area_sqkm": return f"с площадью от {int(min_v)} до {int(max_v)} км²" if min_v is not None and max_v is not None else (f"с площадью не менее {int(min_v)} км²" if min_v is not None else f"с площадью не более {int(max_v)} км²")
    return ""


def _metric_phrase_en(metric, min_v=None, max_v=None):
    if metric == "elevation_m": return f"with elevation from {int(min_v)} to {int(max_v)} m" if min_v is not None and max_v is not None else (f"with elevation at least {int(min_v)} m" if min_v is not None else f"with elevation at most {int(max_v)} m")
    if metric == "length_km": return f"with length from {int(min_v)} to {int(max_v)} km" if min_v is not None and max_v is not None else (f"with length at least {int(min_v)} km" if min_v is not None else f"with length at most {int(max_v)} km")
    if metric == "area_sqkm": return f"with area from {int(min_v)} to {int(max_v)} sq km" if min_v is not None and max_v is not None else (f"with area at least {int(min_v)} sq km" if min_v is not None else f"with area at most {int(max_v)} sq km")
    return ""


def _kind_ru(kind): return GEO_KIND_CFG[kind]["ru_acc"]
def _kind_en(kind): return GEO_KIND_CFG[kind]["en_plural"]

def _constraints_metric(metric, min_v=None, max_v=None):
    if metric == "elevation_m": return {k:v for k,v in {"elevation_min_m": min_v, "elevation_max_m": max_v}.items() if v is not None}
    if metric == "length_km": return {k:v for k,v in {"length_min_km": min_v, "length_max_km": max_v}.items() if v is not None}
    if metric == "area_sqkm": return {k:v for k,v in {"area_min_sqkm": min_v, "area_max_sqkm": max_v}.items() if v is not None}
    return {}


def _ask_type_line(kind, item_var="?item"):
    cfg=GEO_KIND_CFG[kind]
    return f"{item_var} wdt:P31 wd:{cfg['qid']} ." if cfg["direct_only"] else f"{item_var} wdt:P31/wdt:P279* wd:{cfg['qid']} ."


def _ask_metric_lines(kind, min_v=None, max_v=None, item_var="?item", prefix="itemMetric"):
    metric=GEO_KIND_CFG[kind].get("metric"); prop=GEO_KIND_CFG[kind].get("property")
    if not metric or not prop: return ""
    lines = _quantity_required_filter_lines(prop, prefix, metric, ">=", min_v) if min_v is not None else _quantity_required_filter_lines(prop, prefix, metric, "<=", max_v)
    if item_var != "?item": lines = lines.replace("?item ", f"{item_var} ")
    if min_v is not None and max_v is not None: lines += f"\n      FILTER(?{metric} <= {max_v}) ."
    return lines


def _record_from_gold(*, idx, level, template_id, template_family, kind, constraints, q_ru, q_en, gold, ask_body, local_filters, cache, candidate_count, bridge_meta=None, is_advanced=True):
    requested = GEO_CACHE_REQUESTED_BY_LEVEL[level]
    gold_sorted = sorted(_dedupe_entities(gold), key=lambda e: (e.get("label_en") or "", e["qid"]))
    kind_limit_hit = bool(cache.get("kind_audit", {}).get(kind, {}).get("limit_hit"))
    meta = {"source": "wikidata_sparql_cache_v3", "cache_path": str(GEO_CACHE_V3_PATH), "cache_version": cache.get("version"), "match_key": "wikidata_qid", "candidate_count_before_filters": candidate_count, "gold_returned_before_limits": len(gold_sorted), "gold_limit": 300, "gold_returned": len(gold_sorted), "gold_total_before_limit": len(gold_sorted), "gold_truncated_by_local_limit": False, "gold_may_be_incomplete_due_to_cache_limit": kind_limit_hit, "gold_may_be_incomplete_due_to_wdqs_limit": kind_limit_hit, "kind_cache_limit_hit": kind_limit_hit, "constraints_are_cache_backed": True, "template_id": template_id, "template_family": template_family}
    if bridge_meta: meta["bridge_meta"] = bridge_meta
    return {"id": f"geo_int_{level.lower()}_{idx:04d}", "domain": "geo_international", "complexity": level, "query_text_ru": q_ru, "constraints": constraints, "requested_count": requested, "gold_answer_qids": [e["qid"] for e in gold_sorted], "gold_answer_labels_ru": [e.get("label_ru") or e.get("label_en") or e["qid"] for e in gold_sorted], "sparql_query": "# Cache-first generation. Full candidate collection was performed locally from geo_international_cache_v3.\nSELECT DISTINCT ?item WHERE {\n" + ask_body + "\n}", "created_at": _now_iso(), "query_text_en": q_en, "gold_answer_labels_en": [e.get("label_en") or e["qid"] for e in gold_sorted], "is_advanced": is_advanced, "template_id": template_id, "template_family": template_family, "gold_truncated": False, "ask_validator_sparql": "# WDQS validator for the cache-generated geography task.\nASK WHERE {\n      BIND(wd:{ITEM} AS ?item) .\n" + ask_body + "\n}", "local_validator": {"type": "geo_cache_validator", "source": "Wikidata-derived geo cache v3", "match_key": "Wikidata QID", "applies_after": "ask_validator_sparql", "filters": local_filters, "label_matching_used": False, "note": "Gold is collected locally from geo_international_cache_v3. The ASK validator checks the Wikidata structure; local_validator describes cache-backed numeric and bridge filters."}, "gold_collection_meta": meta, "gold_answer_imdb_ids": [], "gold_answer_imdb_titles": []}


def _valid_gold_for_level(level, gold, kind, cache):
    """Return whether a candidate gold set is usable for this level.

    Important: a global cache for a kind can hit its broad collection limit,
    but rejecting every candidate from that kind makes L4/L5 generation impossible.
    We therefore do NOT reject on kind_cache_limit_hit when
    REQUIRE_CACHE_COMPLETE_FOR_GOLD is False. The possible incompleteness is
    recorded explicitly in gold_collection_meta.
    """
    n = len(_dedupe_entities(gold))
    if n < GEO_CACHE_MIN_GOLD_BY_LEVEL[level] or n > GEO_CACHE_MAX_GOLD_BY_LEVEL[level]:
        return False
    if REQUIRE_CACHE_COMPLETE_FOR_GOLD and cache.get("kind_audit", {}).get(kind, {}).get("limit_hit"):
        return False
    return True


def _pick_metric(kind, level, rng):
    metric = GEO_KIND_CFG[kind].get("metric")
    if not metric: return None, None, None
    if metric == "elevation_m":
        if level == "L1": return metric, None, None
        if level == "L5" and rng.random() < 0.35: return metric, rng.choice([1000,1500,2000,3000]), rng.choice([4000,5000,6000,8000])
        return (metric, rng.choice([500,1000,1500,2000,3000,4000,5000]), None) if rng.random() < 0.55 else (metric, None, rng.choice([500,1000,1500,2000,3000]))
    if metric == "length_km":
        if level == "L5" and rng.random() < 0.35: return metric, rng.choice([50,100,250]), rng.choice([500,1000,2000])
        return (metric, rng.choice([50,100,250,500,1000,2000]), None) if rng.random() < 0.55 else (metric, None, rng.choice([50,100,250,500,1000]))
    if metric == "area_sqkm":
        if level == "L5" and rng.random() < 0.35: return metric, rng.choice([50,100,500]), rng.choice([1000,5000,10000])
        return (metric, rng.choice([50,100,500,1000,5000,10000]), None) if rng.random() < 0.55 else (metric, None, rng.choice([50,100,500,1000,5000]))
    return metric, None, None

# Template builders follow. They are intentionally compact and cache-only.

def _tpl_l1_country_continent(cache, idx, level, rng):
    indexes=cache["_indexes"]; kind=rng.choice(["mountain","volcano","lake","waterfall","desert","sea"]); source=indexes["by_kind"][kind]
    if rng.random()<0.55:
        e=rng.choice([x for x in source if x.get("country_qids")]); q=rng.choice(e["country_qids"]); gold=indexes["kind_country"].get((kind,q),[]); constraints={"kind":kind,"country":_label_en(cache,q)}; q_ru=f"Назови 5 {_kind_ru(kind)}, расположенных в {_label_ru(cache,q)}."; q_en=f"Name 5 {_kind_en(kind)} located in {_label_en(cache,q)}."; ask=f"      {_ask_type_line(kind)}\n      ?item wdt:P17 wd:{q} ."
    else:
        e=rng.choice([x for x in source if x.get("continent_qids")]); q=rng.choice(e["continent_qids"]); gold=indexes["kind_continent"].get((kind,q),[]); constraints={"kind":kind,"continent":_label_en(cache,q)}; q_ru=f"Назови 5 {_kind_ru(kind)}, расположенных в {_label_ru(cache,q)}."; q_en=f"Name 5 {_kind_en(kind)} located in {_label_en(cache,q)}."; ask=f"      {_ask_type_line(kind)}\n      ?item wdt:P30 wd:{q} ."
    if not _valid_gold_for_level(level,gold,kind,cache): return None
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_l1_country_continent_kind",template_family="country_continent_kind",kind=kind,constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(gold),is_advanced=False)


def _tpl_l2_country_continent_metric(cache, idx, level, rng):
    indexes=cache["_indexes"]; kind=rng.choice(["mountain","volcano","river","lake","island","desert","sea"]); metric,min_v,max_v=_pick_metric(kind,level,rng)
    if not metric: return None
    source=indexes["by_kind"][kind]
    if rng.random()<0.55:
        e=rng.choice([x for x in source if x.get("country_qids")]); q=rng.choice(e["country_qids"]); base=indexes["kind_country"].get((kind,q),[]); geo_key="country"; geo_en=_label_en(cache,q); geo_ru=_label_ru(cache,q); ask_geo=f"?item wdt:P17 wd:{q} ."
    else:
        e=rng.choice([x for x in source if x.get("continent_qids")]); q=rng.choice(e["continent_qids"]); base=indexes["kind_continent"].get((kind,q),[]); geo_key="continent"; geo_en=_label_en(cache,q); geo_ru=_label_ru(cache,q); ask_geo=f"?item wdt:P30 wd:{q} ."
    gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":kind,geo_key:geo_en,**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,kind,cache): return None
    q_ru=f"Назови 5 {_kind_ru(kind)}, расположенных в {geo_ru}, {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name 5 {_kind_en(kind)} located in {geo_en}, {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line(kind)}\n      {ask_geo}\n"+_ask_metric_lines(kind,min_v,max_v)
    return _record_from_gold(idx=idx,level=level,template_id=f"geo_cache_l2_{geo_key}_{kind}_{metric}",template_family="country_continent_metric",kind=kind,constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),is_advanced=False)

# Generic bridge templates.
def _tpl_mountain_range(cache, idx, level, rng):
    indexes=cache["_indexes"]; seed=_choose_seed_with_values(rng,indexes["by_kind"]["mountain"],"mountain_range_qids",metric="elevation_m")
    if not seed: return None
    range_qid=rng.choice(seed["mountain_range_qids"]); metric="elevation_m"; min_v,max_v=_pick_metric("mountain",level,rng)[1:]
    if level in {"L3","L4"} and min_v is None: min_v=rng.choice([1000,2000,3000,4000,5000,6000,7000,8000]); max_v=None
    base=[e for e in indexes["mountain_range"].get(range_qid,[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":"mountain","mountain_range_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,"mountain",cache): return None
    q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} горы, из того же горного хребта, что и «{seed['label_ru']}», {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} mountains from the same mountain range as \"{seed['label_en']}\", {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line('mountain')}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P4552 ?bridgeRange .\n      ?item wdt:P4552 ?bridgeRange .\n      FILTER(?item != ?seed) .\n"+_ask_metric_lines('mountain',min_v,max_v); bm={"bridge":"mountain_range","constraint_key":"mountain_range_from_object","wikidata_property":"P4552","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_mountain_same_range_elevation",template_family="mountains",kind="mountain",constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)


def _tpl_river_same_mouth(cache, idx, level, rng):
    indexes=cache["_indexes"]; seed=_choose_seed_with_values(rng,indexes["by_kind"]["river"],"mouth_body_qids",metric="length_km")
    if not seed: return None
    mb=rng.choice(seed["mouth_body_qids"]); metric,min_v,max_v=_pick_metric("river",level,rng); base=[e for e in indexes["mouth_body"].get(mb,[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":"river","mouth_body_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,"river",cache): return None
    q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} рек, впадающих в тот же водоём, что и «{seed['label_ru']}», {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} rivers flowing into the same body of water as \"{seed['label_en']}\", {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line('river')}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P403 ?bridgeMouth .\n      ?item wdt:P403 ?bridgeMouth .\n      FILTER(?item != ?seed) .\n"+_ask_metric_lines('river',min_v,max_v); bm={"bridge":"mouth_body","constraint_key":"mouth_body_from_object","wikidata_property":"P403","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_river_same_mouth_length",template_family="river_mouth",kind="river",constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)


def _tpl_basin_country(cache, idx, level, rng, kind="river"):
    indexes=cache["_indexes"]; seed=_choose_seed_with_values(rng,indexes["by_kind"][kind],"basin_country_qids",metric=GEO_KIND_CFG[kind].get("metric"))
    if not seed: return None
    bc=rng.choice(seed["basin_country_qids"]); metric,min_v,max_v=_pick_metric(kind,level,rng); base=[e for e in indexes["basin_country_kind"].get((kind,bc),[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":kind,"basin_country_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,kind,cache): return None
    q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_ru(kind)}, имеющих одну из тех же стран водосборного бассейна, что и «{seed['label_ru']}», {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_en(kind)} sharing one of the same basin countries as \"{seed['label_en']}\", {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line(kind)}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P205 ?bridgeBasinCountry .\n      ?item wdt:P205 ?bridgeBasinCountry .\n      FILTER(?item != ?seed) .\n"+_ask_metric_lines(kind,min_v,max_v); bm={"bridge":"basin_country","constraint_key":"basin_country_from_object","wikidata_property":"P205","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id=f"geo_cache_{kind}_same_basin_country_metric",template_family=f"{kind}_basin_country",kind=kind,constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)


def _tpl_lake_outflow(cache, idx, level, rng):
    indexes=cache["_indexes"]; seed=_choose_seed_with_values(rng,indexes["by_kind"]["lake"],"outflow_qids",metric="area_sqkm")
    if not seed: return None
    of=rng.choice(seed["outflow_qids"]); metric,min_v,max_v=_pick_metric("lake",level,rng); base=[e for e in indexes["outflow"].get(of,[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":"lake","outflow_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,"lake",cache): return None
    q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} озера, имеющих тот же сток, что и «{seed['label_ru']}», {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} lakes with the same outflow as \"{seed['label_en']}\", {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line('lake')}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P201 ?bridgeOutflow .\n      ?item wdt:P201 ?bridgeOutflow .\n      FILTER(?item != ?seed) .\n"+_ask_metric_lines('lake',min_v,max_v); bm={"bridge":"outflow","constraint_key":"outflow_from_object","wikidata_property":"P201","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_lake_same_outflow_area",template_family="lake_outflow",kind="lake",constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)


def _tpl_island_water_body(cache, idx, level, rng):
    indexes=cache["_indexes"]; seed=_choose_seed_with_values(rng,indexes["by_kind"]["island"],"water_body_qids",metric="area_sqkm")
    if not seed: return None
    wb=rng.choice(seed["water_body_qids"]); metric,min_v,max_v=_pick_metric("island",level,rng); base=[e for e in indexes["water_body"].get(wb,[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v); constraints={"kind":"island","water_body_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,"island",cache): return None
    q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} острова, расположенных в том же водоёме или у того же водоёма, что и «{seed['label_ru']}», {_metric_phrase_ru(metric,min_v,max_v)}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} islands located in or next to the same body of water as \"{seed['label_en']}\", {_metric_phrase_en(metric,min_v,max_v)}."; ask=f"      {_ask_type_line('island')}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P206 ?bridgeWaterBody .\n      ?item wdt:P206 ?bridgeWaterBody .\n      FILTER(?item != ?seed) .\n"+_ask_metric_lines('island',min_v,max_v); bm={"bridge":"water_body","constraint_key":"water_body_from_object","wikidata_property":"P206","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_island_same_water_body_area",template_family="island_water_body",kind="island",constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)



def _tpl_cross_kind_country_contains(cache, idx, level, rng):
    """Mixed-geo country bridge with UNION semantics over all matching countries.

    Important semantic fix:
    - Old version selected one hidden country and collected answers only there, while the
      text said "located in a country that has ...". That made gold incomplete/incorrect.
    - New version does NOT choose a hidden country. It collects the union of answers from
      every country that contains a related object satisfying the related metric.
    - For L4/L5 it may add a visible continent constraint to keep the gold set focused;
      the continent is shown in query_text and constraints, so it is not hidden.
    """
    if level not in {"L3", "L4", "L5"}:
        return None

    indexes = cache["_indexes"]

    # Pairs are intentionally cross-kind and user-like. The related object is the hidden bridge.
    patterns = [
        # answer_kind, related_kind, related_metric, related_min, related_max
        ("river", "desert", "area_sqkm", 500, None),
        ("river", "desert", "area_sqkm", None, 5000),
        ("river", "mountain", "elevation_m", 3000, None),
        ("lake", "mountain", "elevation_m", 3000, None),
        ("waterfall", "volcano", "elevation_m", 2000, None),
        ("island", "river", "length_km", 500, None),
        ("mountain", "lake", "area_sqkm", 500, None),
        ("lake", "desert", "area_sqkm", 1000, None),
    ]

    answer_kind, related_kind, rel_metric, rel_min, rel_max = rng.choice(patterns)

    # For L5, prefer tighter answer-side ranges so the union across countries remains hard.
    ans_metric, ans_min, ans_max = _pick_metric(answer_kind, level, rng)
    if not ans_metric and answer_kind != "waterfall":
        return None

    # Make L5 mixed patterns more constrained and interesting.
    if level == "L5" and ans_metric == "length_km" and rng.random() < 0.70:
        lo = rng.choice([50, 100, 250])
        hi = rng.choice([500, 1000, 2000])
        if hi <= lo:
            hi = lo + 500
        ans_min, ans_max = lo, hi
    elif level == "L5" and ans_metric == "area_sqkm" and rng.random() < 0.70:
        lo = rng.choice([50, 100, 500])
        hi = rng.choice([1000, 5000, 10000])
        if hi <= lo:
            hi = lo + 1000
        ans_min, ans_max = lo, hi
    elif level == "L5" and ans_metric == "elevation_m" and rng.random() < 0.70:
        lo = rng.choice([500, 1000, 2000])
        hi = rng.choice([3000, 5000, 7000])
        if hi <= lo:
            hi = lo + 1000
        ans_min, ans_max = lo, hi

    def _related_phrase_ru(kind: str) -> str:
        return {
            "desert": "пустыня",
            "mountain": "гора",
            "volcano": "вулкан",
            "river": "река",
            "lake": "озеро",
            "island": "остров",
            "waterfall": "водопад",
            "sea": "море",
        }.get(kind, _kind_ru(kind))

    def _collect_union(continent_qid=None):
        union_gold = []
        matched_countries = []
        matched_related_count = 0
        answer_candidates_before_metric = 0

        for country_qid, by_kind in indexes["country_contains"].items():
            related_pool = by_kind.get(related_kind, [])
            related_pass = _filter_metric(related_pool, rel_metric, min_v=rel_min, max_v=rel_max)
            if not related_pass:
                continue

            ans_pool = list(by_kind.get(answer_kind, []))
            if continent_qid is not None:
                ans_pool = [e for e in ans_pool if continent_qid in set(e.get("continent_qids", []))]
            if not ans_pool:
                continue

            answer_candidates_before_metric += len(ans_pool)
            ans_pass = _filter_metric(ans_pool, ans_metric, min_v=ans_min, max_v=ans_max) if ans_metric else ans_pool
            if not ans_pass:
                continue

            matched_countries.append(country_qid)
            matched_related_count += len(related_pass)
            union_gold.extend(ans_pass)

        return {
            "continent_qid": continent_qid,
            "gold": _dedupe_entities(union_gold),
            "matched_countries": matched_countries,
            "matched_related_count": matched_related_count,
            "candidate_count": answer_candidates_before_metric,
        }

    # Candidate scopes: global union plus visible continent-constrained unions.
    all_continents = sorted({c for e in indexes["by_kind"].get(answer_kind, []) for c in e.get("continent_qids", [])})
    scopes = []
    if level == "L3":
        scopes = [None] + rng.sample(all_continents, k=min(len(all_continents), 3))
    elif level == "L4":
        scopes = rng.sample(all_continents, k=min(len(all_continents), 5)) + [None]
    else:
        # L5 should usually be visibly narrower; global union is tried last.
        scopes = rng.sample(all_continents, k=min(len(all_continents), 7)) + ([None] if rng.random() < 0.25 else [])

    options = []
    for cont in scopes:
        opt = _collect_union(cont)
        if _valid_gold_for_level(level, opt["gold"], answer_kind, cache):
            options.append(opt)

    if not options:
        return None

    # Prefer compact but non-trivial gold sets for hard levels.
    def _score(opt):
        n = len(opt["gold"])
        target = 10 if level == "L5" else 16 if level == "L4" else 22
        continent_bonus = 0 if opt["continent_qid"] is not None else 8
        return abs(n - target) + continent_bonus + rng.random()

    chosen = sorted(options, key=_score)[0]
    gold = chosen["gold"]
    continent_qid = chosen["continent_qid"]

    constraints = {
        "kind": answer_kind,
        "country_contains_kind": related_kind,
        "country_scope": "all_matching_countries" if continent_qid is None else "matching_countries_on_continent",
        **_constraints_metric(ans_metric, ans_min, ans_max),
    }
    if continent_qid is not None:
        constraints["continent"] = _label_en(cache, continent_qid)

    if rel_metric == "area_sqkm":
        constraints[f"related_{related_kind}_area_min_sqkm" if rel_min is not None else f"related_{related_kind}_area_max_sqkm"] = rel_min if rel_min is not None else rel_max
    elif rel_metric == "elevation_m":
        constraints[f"related_{related_kind}_elevation_min_m" if rel_min is not None else f"related_{related_kind}_elevation_max_m"] = rel_min if rel_min is not None else rel_max
    elif rel_metric == "length_km":
        constraints[f"related_{related_kind}_length_min_km" if rel_min is not None else f"related_{related_kind}_length_max_km"] = rel_min if rel_min is not None else rel_max

    ans_ru = (", " + _metric_phrase_ru(ans_metric, ans_min, ans_max)) if ans_metric else ""
    ans_en = (", " + _metric_phrase_en(ans_metric, ans_min, ans_max)) if ans_metric else ""
    rel_ru = _metric_phrase_ru(rel_metric, rel_min, rel_max)
    rel_en = _metric_phrase_en(rel_metric, rel_min, rel_max)

    scope_ru = ""
    scope_en = ""
    if continent_qid is not None:
        scope_ru = f", расположенных в {_label_ru(cache, continent_qid)}"
        scope_en = f", located in {_label_en(cache, continent_qid)}"

    q_ru = (
        f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_ru(answer_kind)}{ans_ru}{scope_ru}, "
        f"расположенных в стране, где есть {_related_phrase_ru(related_kind)} {rel_ru}."
    )
    q_en = (
        f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_en(answer_kind)}{ans_en}{scope_en}, "
        f"located in a country that has a {_kind_en(related_kind).rstrip('s')} {rel_en}."
    )

    # ASK mirrors the natural-language semantics: union over all countries that contain
    # a related object satisfying the related constraint. There is no hidden selected country.
    ask = (
        f"      {_ask_type_line(answer_kind)}\n"
        f"      ?item wdt:P17 ?bridgeCountry .\n"
        f"      ?related wdt:P17 ?bridgeCountry .\n"
        f"      {_ask_type_line(related_kind, '?related')}\n"
    )
    if continent_qid is not None:
        ask += (
            f"      {{ ?item wdt:P30 wd:{continent_qid} . }}\n"
            f"      UNION\n"
            f"      {{ ?item wdt:P17 ?itemCountryForContinent . ?itemCountryForContinent wdt:P30 wd:{continent_qid} . }}\n"
        )
    if ans_metric:
        ask += _ask_metric_lines(answer_kind, ans_min, ans_max) + "\n"
    ask += _ask_metric_lines(related_kind, rel_min, rel_max, item_var="?related", prefix="relatedMetric")

    bm = {
        "bridge": f"country_contains_{related_kind}",
        "constraint_key": "country_contains_kind",
        "semantics": "union_all_countries: answer and related object share any country satisfying the related constraint; no hidden single country is selected",
        "intermediate_value_hidden_in_query": True,
        "country_scope": constraints["country_scope"],
        "continent_qid": continent_qid,
        "continent_label_en": _label_en(cache, continent_qid) if continent_qid else None,
        "related_kind": related_kind,
        "related_metric": rel_metric,
        "related_metric_min": rel_min,
        "related_metric_max": rel_max,
        "matching_country_count": len(chosen["matched_countries"]),
        "matching_country_qids_preview": chosen["matched_countries"][:25],
        "matched_related_count_in_cache": chosen["matched_related_count"],
    }

    return _record_from_gold(
        idx=idx,
        level=level,
        template_id=f"geo_cache_mixed_{answer_kind}_country_contains_{related_kind}_union",
        template_family="mixed_country_contains_union",
        kind=answer_kind,
        constraints=constraints,
        q_ru=q_ru,
        q_en=q_en,
        gold=gold,
        ask_body=ask,
        local_filters=constraints,
        cache=cache,
        candidate_count=chosen["candidate_count"],
        bridge_meta=bm,
    )


def _tpl_same_country_metric(cache, idx, level, rng):
    indexes=cache["_indexes"]; seed_kind=rng.choice(["lake","river","mountain","volcano","desert"]); answer_kind=rng.choice([k for k in ["mountain","volcano","river","lake","island","waterfall","desert"] if k!=seed_kind]); seed=_choose_seed_with_values(rng,indexes["by_kind"][seed_kind],"country_qids")
    if not seed: return None
    country=rng.choice(seed["country_qids"]); metric,min_v,max_v=_pick_metric(answer_kind,level,rng); base=[e for e in indexes["kind_country"].get((answer_kind,country),[]) if e["qid"]!=seed["qid"]]; gold=_filter_metric(base,metric,min_v=min_v,max_v=max_v) if metric else base; constraints={"kind":answer_kind,"country_from_object":seed["label_en"],**_constraints_metric(metric,min_v,max_v)}
    if not _valid_gold_for_level(level,gold,answer_kind,cache): return None
    mru=(", "+_metric_phrase_ru(metric,min_v,max_v)) if metric else ""; men=(", "+_metric_phrase_en(metric,min_v,max_v)) if metric else ""; q_ru=f"Назови {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_ru(answer_kind)}, расположенных в одной из стран, где находится «{seed['label_ru']}»{mru}."; q_en=f"Name {GEO_CACHE_REQUESTED_BY_LEVEL[level]} {_kind_en(answer_kind)} located in one of the countries of \"{seed['label_en']}\"{men}."; ask=f"      {_ask_type_line(answer_kind)}\n      BIND(wd:{seed['qid']} AS ?seed) .\n      ?seed wdt:P17 ?bridgeCountry .\n      ?item wdt:P17 ?bridgeCountry .\n      FILTER(?item != ?seed) .\n"
    if metric: ask+=_ask_metric_lines(answer_kind,min_v,max_v)
    bm={"bridge":"country","constraint_key":"country_from_object","wikidata_property":"P17","seed_label_en":seed['label_en'],"seed_label_ru":seed['label_ru'],"seed_kind":seed_kind,"semantics":"any_shared_value","intermediate_value_hidden_in_query":True}
    return _record_from_gold(idx=idx,level=level,template_id="geo_cache_cross_kind_same_country_metric",template_family="same_country_cross_kind",kind=answer_kind,constraints=constraints,q_ru=q_ru,q_en=q_en,gold=gold,ask_body=ask,local_filters=constraints,cache=cache,candidate_count=len(base),bridge_meta=bm)

GEO_CACHE_TEMPLATE_REGISTRY = {
    "L1": [(_tpl_l1_country_continent, 1.0)],
    "L2": [(_tpl_l2_country_continent_metric, 1.0)],
    "L3": [(_tpl_mountain_range, 1.5), (_tpl_river_same_mouth, 1.4), (lambda c,i,l,r: _tpl_basin_country(c,i,l,r,"river"), 0.7), (lambda c,i,l,r: _tpl_basin_country(c,i,l,r,"lake"), 0.6), (_tpl_lake_outflow, 1.2), (_tpl_island_water_body, 1.2), (_tpl_cross_kind_country_contains, 1.8), (_tpl_same_country_metric, 1.0)],
    "L4": [(_tpl_mountain_range, 1.3), (_tpl_river_same_mouth, 1.5), (lambda c,i,l,r: _tpl_basin_country(c,i,l,r,"river"), 0.5), (lambda c,i,l,r: _tpl_basin_country(c,i,l,r,"lake"), 0.5), (_tpl_lake_outflow, 1.3), (_tpl_island_water_body, 1.4), (_tpl_cross_kind_country_contains, 2.3), (_tpl_same_country_metric, 1.0)],
    "L5": [(_tpl_mountain_range, 1.2), (_tpl_river_same_mouth, 1.4), (_tpl_lake_outflow, 1.4), (_tpl_island_water_body, 1.5), (_tpl_cross_kind_country_contains, 2.6), (_tpl_same_country_metric, 0.8)],
}


def _read_jsonl(path):
    if not path.exists(): return []
    out=[]
    with path.open("r",encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try: out.append(json.loads(line))
                except Exception: pass
    return out

def _append_jsonl(path, rec):
    with path.open("a",encoding="utf-8") as f: f.write(json.dumps(rec,ensure_ascii=False)+"\n"); f.flush()

def _write_json(path, obj):
    with path.open("w",encoding="utf-8") as f: json.dump(obj,f,ensure_ascii=False,indent=2); f.flush()

def _rec_key(r): return (r.get("query_text_ru"), json.dumps(r.get("constraints",{}),ensure_ascii=False,sort_keys=True))

def _weighted_template(level,rng):
    entries=GEO_CACHE_TEMPLATE_REGISTRY[level]; return rng.choices([x[0] for x in entries], weights=[x[1] for x in entries], k=1)[0]

def _current_geo_cache_audit(records, skipped, cache):
    return {"domain":"geo_international","mode":"cache_first_v3","target_plan":TARGET_PLAN_GEO_CACHE,"records_total":len(records),"counts_by_complexity":dict(Counter(r.get("complexity") for r in records)),"counts_by_kind":dict(Counter((r.get("constraints") or {}).get("kind") for r in records)),"counts_by_template_family":dict(Counter(r.get("template_family") for r in records)),"skipped_count":len(skipped),"skipped_preview":skipped[-100:],"cache_path":str(GEO_CACHE_V3_PATH),"cache_kind_audit":cache.get("kind_audit",{}),"updated_at":_now_iso()}

def generate_geo_from_cache_v3():
    cache=build_geo_cache_v3(rebuild=REBUILD_GEO_CACHE_V3); cache["_indexes"]=_make_geo_indexes(cache)
    print("cache loaded:", GEO_CACHE_V3_PATH.resolve())
    print("cache kind counts:", {k:len(v) for k,v in cache["kinds"].items()})
    print("cache limit hits:", {k:a.get("limit_hit") for k,a in cache.get("kind_audit",{}).items()})
    records=_read_jsonl(GEO_CACHE_OUT_PATH); seen={_rec_key(r) for r in records}; counts=Counter(r.get("complexity") for r in records); skipped=[]; idx=len(records)+1
    if records:
        print("existing output records:", len(records), "counts:", dict(counts))
        print("NOTE: delete geo_international.jsonl/checkpoint/audit first if you want a clean L4-L5-only file.")
    total_target=sum(TARGET_PLAN_GEO_CACHE.values()); initial=sum(min(counts.get(lvl,0),target) for lvl,target in TARGET_PLAN_GEO_CACHE.items()); overall=tqdm(total=total_target, initial=initial, desc="geo cache total") if tqdm else None
    try:
        for level,target in TARGET_PLAN_GEO_CACHE.items():
            ok=counts.get(level,0)
            if ok>=target: print(f"SKIP {level}: already {ok}/{target}"); continue
            attempts=0; level_bar=tqdm(total=target, initial=ok, desc=f"geo_cache:{level}", leave=True) if tqdm else None
            while ok<target and attempts<GEO_CACHE_MAX_ATTEMPTS_BY_LEVEL[level]:
                attempts+=1; fn=_weighted_template(level,GEO_CACHE_RNG)
                try:
                    rec=fn(cache,idx,level,GEO_CACHE_RNG)
                    if rec is None: skipped.append({"complexity":level,"reason":"template_returned_none","template":getattr(fn,"__name__",str(fn))}); continue
                    key=_rec_key(rec)
                    if key in seen: skipped.append({"complexity":level,"reason":"duplicate","template_family":rec.get("template_family")}); continue
                    seen.add(key); records.append(rec); _append_jsonl(GEO_CACHE_OUT_PATH,rec); idx+=1; ok+=1; counts[level]+=1
                    audit=_current_geo_cache_audit(records,skipped,cache); _write_json(GEO_CACHE_AUDIT_PATH,audit); _write_json(GEO_CACHE_CHECKPOINT_PATH,{"last_record":rec,"audit":audit})
                    postfix={"ok":ok,"attempts":attempts,"kind":rec["constraints"].get("kind"),"gold":len(rec.get("gold_answer_qids",[])),"family":rec.get("template_family")}
                    if level_bar: level_bar.update(1); level_bar.set_postfix(postfix)
                    if overall: overall.update(1); overall.set_postfix(postfix)
                except KeyboardInterrupt: raise
                except Exception as e: skipped.append({"complexity":level,"reason":type(e).__name__,"error":str(e)[:1000],"template":getattr(fn,"__name__",str(fn))})
            if level_bar: level_bar.close()
            print(f"OK geo_cache:{level} {ok}/{target} generated in {attempts} attempts")
    finally:
        if overall: overall.close()
        audit=_current_geo_cache_audit(records,skipped,cache); _write_json(GEO_CACHE_AUDIT_PATH,audit)
        print("saved:",GEO_CACHE_OUT_PATH.resolve()); print("audit:",GEO_CACHE_AUDIT_PATH.resolve()); print("records:",len(records)); print("counts:",dict(Counter(r.get("complexity") for r in records))); print("kinds:",dict(Counter((r.get("constraints") or {}).get("kind") for r in records))); print("families:",dict(Counter(r.get("template_family") for r in records))); print("skipped:",len(skipped))

if RUN_GEO_CACHE_FIRST_GENERATION:
    generate_geo_from_cache_v3()
else:
    print("RUN_GEO_CACHE_FIRST_GENERATION is False; cache-first generation did not run.")


loading existing geo cache: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/cache/geo_international_cache_v3.json
cache loaded: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/cache/geo_international_cache_v3.json
cache kind counts: {'mountain': 194, 'volcano': 86, 'river': 2309, 'lake': 2135, 'island': 167, 'desert': 472, 'waterfall': 8320, 'sea': 2}
cache limit hits: {'mountain': True, 'volcano': True, 'river': True, 'lake': True, 'island': True, 'desert': True, 'waterfall': True, 'sea': True}
existing output records: 95 counts: {'L1': 15, 'L2': 25, 'L3': 30, 'L4': 25}
NOTE: delete geo_international.jsonl/checkpoint/audit first if you want a clean L4-L5-only file.


geo cache total:  42%|████▏     | 25/60 [00:00<?, ?it/s]

SKIP L1: already 15/0
SKIP L2: already 25/0
SKIP L3: already 30/0
SKIP L4: already 25/25




geo cache total:  52%|█████▏    | 31/60 [00:00<00:00, 59.40it/s, ok=6, attempts=14, kind=lake, gold=10, family=mixed_country_contains_union]

geo cache total:  62%|██████▏   | 37/60 [00:00<00:00, 51.56it/s, ok=12, attempts=31, kind=lake, gold=9, family=mixed_country_contains_union] 

geo cache total:  72%|███████▏  | 43/60 [00:00<00:00, 50.44it/s, ok=18, attempts=39, kind=island, gold=13, family=island_water_body]

geo cache total:  83%|████████▎ | 50/60 [00:00<00:00, 53.00it/s, ok=25, attempts=52, kind=river, gold=5, family=river_mouth]       

geo cache total:  93%|█████████▎| 56/60 [00:00<00:00, 48.16it/s, ok=31, attempts=60, kind=lake, gold=6, family=mixed_country_contains_union]

geo cache total: 100%|██████████| 60/60 [00:00<00:00, 44.91it/s, ok=35, attempts=69, kind=lake, gold=4, family=mixed_country_contains_union]

OK geo_cache:L5 35/35 generated in 69 attempts
saved: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/geo_international.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/geo_international_generation_audit.json
records: 130
counts: {'L1': 15, 'L2': 25, 'L3': 30, 'L4': 25, 'L5': 35}
kinds: {'waterfall': 5, 'desert': 7, 'sea': 3, 'lake': 31, 'volcano': 2, 'mountain': 24, 'river': 37, 'island': 21}
families: {'continent_kind': 22, 'country_kind': 18, 'river_basin_country': 18, 'lake_basin_country': 10, 'same_country': 6, 'same_continent': 5, 'mountains': 8, 'cross_kind_same_country': 2, 'rivers': 1, 'mountain_range_hard': 3, 'lake_outflow': 1, 'same_continent_hard': 3, 'mixed_country_contains_union': 24, 'island_water_body': 8, 'river_mouth': 1}
skipped: 34
